# Forklift Feature-Based Anomaly Detection PoC

`data/movie_preprocess/normal` にある正常動画だけを使い、音声特徴・オプティカルフロー特徴・グローバルモーション特徴から正常分布を学習する PoC ノートブックです。

今回の初期実装ではセグメンテーションや RGB 画像そのものの深層学習は扱わず、CPU で動かしやすい OpenCV + librosa + scikit-learn の特徴量ベース構成にします。

## 0. パッケージ確認

必要なパッケージが未導入の場合は、下のコメントを外してインストールしてください。音声抽出済みの `*_audio.wav` を使うため、通常は `ffmpeg` がなくても学習まで進められます。

In [1]:
%pip install -q opencv-python numpy pandas matplotlib scipy librosa scikit-learn tqdm joblib

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Note: you may need to restart the kernel to use updated packages.


## 1. ライブラリ読み込み

In [2]:
from __future__ import annotations

import json
import math
import os
import shutil
import subprocess
import tempfile
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import resample_poly
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    import librosa
except Exception as exc:
    librosa = None
    warnings.warn(f"librosa could not be imported. Audio features will use a scipy fallback. reason={exc}")

plt.rcParams["figure.figsize"] = (14, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.max_columns", 120)

## 2. 設定値

最初は `MAX_TRAIN_VIDEOS` を小さめにして特徴量の妥当性と処理時間を確認し、問題なければ `None` にして正常データ全体で学習します。

In [3]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root whether the notebook is run from repo root or notebooks/."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "movie_preprocess").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find repository root from {start}. "
        "Run this notebook inside the forklift repository."
    )

PROJECT_ROOT = find_project_root()
TRAIN_DATA_DIR = PROJECT_ROOT / "data" / "movie_preprocess" / "normal"
MANIFEST_PATH = PROJECT_ROOT / "data" / "movie_preprocess" / "movie_preprocess_manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "anomaly_feature_poc"
TRAIN_SAMPLE_LIST_PATH = PROJECT_ROOT / "data" / "train_sample_list.csv"

FPS_SAMPLE = 10
WINDOW_SEC = 0.2
AUDIO_SR = 16000
N_MELS = 16

USE_FRONT = True
USE_REAR = True
FRONT_REAR_SPLIT = "vertical"
FRONT_IS_TOP = True

FRAME_RESIZE_WIDTH = 480
FLOW_ANALYSIS_SCALE = 0.75  # Farneback flowだけ少し縮小した画像で測る。0.5〜0.75程度で調整。
FLOW_GRID = (3, 3)
FLOW_MIN_VALID_CELL_RATIO = 0.01
FLOW_SCORE_LOWER_QUANTILE = 0.50
FLOW_SCORE_UPPER_QUANTILE = 0.99
FLOW_CELL_SCORE_UPPER_QUANTILE = 0.95
FLOW_EVENT_THRESHOLD = 0.65
FLOW_DURATION_ALPHA = 1.0
FLOW_GLOBALITY_WEIGHT = 0.5
FLOW_TRANSIENCE_WEIGHT = 0.3
FLOW_CHANGE_AMOUNT_WEIGHT = 0.2
FLOW_GLOBALITY_RATIO_WEIGHT = 0.85
FLOW_GLOBALITY_INTENSITY_WEIGHT = 0.15
FLOW_XY_IMPACT_CONFIG = {
    "grid": FLOW_GRID,
    "min_valid_cell_ratio": FLOW_MIN_VALID_CELL_RATIO,
    "score_lower_quantile": FLOW_SCORE_LOWER_QUANTILE,
    "score_upper_quantile": FLOW_SCORE_UPPER_QUANTILE,
    "cell_score_upper_quantile": FLOW_CELL_SCORE_UPPER_QUANTILE,
    "event_threshold": FLOW_EVENT_THRESHOLD,
    "duration_alpha": FLOW_DURATION_ALPHA,
    "globality_weight": FLOW_GLOBALITY_WEIGHT,
    "transience_weight": FLOW_TRANSIENCE_WEIGHT,
    "change_amount_weight": FLOW_CHANGE_AMOUNT_WEIGHT,
    "globality_ratio_weight": FLOW_GLOBALITY_RATIO_WEIGHT,
    "globality_intensity_weight": FLOW_GLOBALITY_INTENSITY_WEIGHT,
}
FLOW_XY_IMPACT_COLUMNS = [
    "front_flow_change_amount_score",
    "front_flow_globality_score",
    "front_flow_transience_score",
    "front_flow_xy_impact_score",
    "rear_flow_change_amount_score",
    "rear_flow_globality_score",
    "rear_flow_transience_score",
    "rear_flow_xy_impact_score",
]
MAX_CORNERS = 200
RANDOM_STATE = 42

# 初回確認用。全件学習するときは None に変更してください。
MAX_TRAIN_VIDEOS = 35

# Isolation Forest の contamination は「異常候補として上位何割を強めに見るか」の目安です。
CONTAMINATION = "auto"
TOP_N_ANOMALIES = 8

# 広域振動スコア用の窓設定。各カメラで同時刻の上位グリッドを平均して motion モデルへ入れます。
FLOW_SCORE_WINDOW_SEC = 1.0
FLOW_SCORE_HOP_SEC = 0.5
FLOW_SCORE_ALPHA_MIN = 0.04
FLOW_SCORE_ALPHA_MAX = 0.42
FLOW_SCORE_MIN_VISIBLE = 1e-6
FLOW_SCORE_HIGH_RATIO_FRACTION = 0.5
FLOW_DIRECTION_MIN_MAG = 0.025
DIRECTION_JITTER_HIGH_THRESHOLD = 1.5
DIRECTION_JITTER_LOW_THRESHOLD = 0.8
DIRECTION_JITTER_THRESHOLD_MODE = "percentile"
DIRECTION_JITTER_HIGH_PERCENTILE = 85.0
DIRECTION_JITTER_LOW_PERCENTILE = 60.0
DIRECTION_JITTER_LOW_EXPANSION_STEPS = 2
DIRECTION_JITTER_SCORE_LOWER_PERCENTILE = 0.0
DIRECTION_JITTER_SCORE_UPPER_PERCENTILE = 95.0
BROAD_VIBRATION_TOP_K = 5
BROAD_VIBRATION_COLUMNS = [
    "front_broad_vibration_score",
    "rear_broad_vibration_score",
]

# 取得・利用可能な特徴量グループの一覧。
# 実際にどのモデルへ入れるかは SCORE_MODEL_CONFIGS 側で切り替えます。
FEATURE_GROUPS = {
    "audio_basic": True,          # RMS/energy/peak/zcr/spectral centroid など
    "audio_mel": True,            # audio_mel_00.. の log-mel 特徴
    "broad_vibration": True,      # front/rear の広域振動スコア
    "flow_basic": False,          # front/rear の flow magnitude/angle/x/y 統計
    "flow_grid": False,           # 3x3 グリッド別の optical flow 統計
    "flow_change": False,         # 旧 motion 用の flow xy impact スコア（学習入力からは外す）
    "flow_change_diagnostic": False,  # raw差分・高変化セル数・持続時間などの確認用列
    "global_motion": False,       # 旧 motion 用の特徴点追跡特徴（学習入力からは外す）
    "cross_camera": False,        # front/rear の平均・差分・同時ピーク特徴
    "sensor": False,              # motion モデルには広域振動スコアのみを使う
    "context": False,             # motion モデルには広域振動スコアのみを使う
}

# 個別に落としたい列がある場合だけ使います。例: ["audio_mel_00", "front_flow_angle_mean"]
FEATURE_EXCLUDE_COLUMNS: list[str] = []

# 標準化後のモデル入力にかけるグループ別重み。1.0が標準です。
# IsolationForest ではスケール重みの効き方は限定的なので、最終スコアの重みは FINAL_SCORE_WEIGHTS で調整します。
FEATURE_GROUP_WEIGHTS = {
    "audio_basic": 1.0,
    "audio_mel": 1.0,
    "broad_vibration": 1.0,
    "flow_basic": 1.0,
    "flow_grid": 1.0,
    "flow_change": 1.0,
    "flow_change_diagnostic": 1.0,
    "global_motion": 1.0,
    "cross_camera": 1.0,
    "sensor": 1.0,
    "context": 1.0,
}

# モデル別に使う特徴量グループを定義します。
# audio は音声のみ、motion は映像由来の動き + 任意の文脈/センサー特徴を使います。
SCORE_MODEL_CONFIGS = {
    "audio": {
        "enabled": True,
        "score_column": "audio_anomaly_score",
        "raw_score_column": "audio_anomaly_score_raw",
        "feature_groups": {
            "audio_basic": True,
            "audio_mel": True,
            "broad_vibration": False,
            "flow_basic": False,
            "flow_grid": False,
            "flow_change": False,
            "flow_change_diagnostic": False,
            "global_motion": False,
            "cross_camera": False,
            "sensor": False,
            "context": False,
        },
        "feature_group_weights": FEATURE_GROUP_WEIGHTS,
    },
    "motion": {
        "enabled": True,
        "score_column": "motion_anomaly_score",
        "raw_score_column": "motion_anomaly_score_raw",
        "feature_groups": {
            "audio_basic": False,
            "audio_mel": False,
            "broad_vibration": True,
            "flow_basic": False,
            "flow_grid": False,
            "flow_change": False,
            "flow_change_diagnostic": False,
            "global_motion": False,
            "cross_camera": False,
            "sensor": False,
            "context": False,
        },
        "feature_group_weights": FEATURE_GROUP_WEIGHTS,
    },
}

# raw anomaly score を 0..1 に近い比較可能な値へ写像するための、正常学習データ側の分位点。
SCORE_CALIBRATION_QUANTILES = (0.01, 0.99)

# 音声ピークと動きピークの時間同期を見る設定。2窓なら WINDOW_SEC=0.2 秒で前後0.4秒を許容します。
SYNC_SCORE_CONFIG = {
    "audio_score_column": "audio_anomaly_score",
    "motion_score_column": "motion_anomaly_score",
    "max_lag_windows": 2,
}

# 最終スコアの合成重み。音声を強めたい場合は audio_anomaly_score を上げます。
FINAL_SCORE_WEIGHTS = {
    "audio_anomaly_score": 0.45,
    "motion_anomaly_score": 0.35,
    "sync_score": 0.20,
}

# グラフに出す列。スコア系と特徴量系を分けて、後からオン/オフしやすくします。
PLOT_SCORE_COLUMNS = [
    "anomaly_score",
    "audio_anomaly_score",
    "motion_anomaly_score",
    "sync_score",
]
PLOT_FEATURE_COLUMNS = [
    "audio_rms",
    "audio_high_freq_energy",
    "front_broad_vibration_score",
    "rear_broad_vibration_score",
    "front_global_translation_norm",
    "rear_global_translation_norm",
    "front_flow_mag_mean",
    "rear_flow_mag_mean",
    "front_flow_x_mean",
    "rear_flow_x_mean",
    "front_flow_y_mean",
    "rear_flow_y_mean",
    "front_flow_change_amount_score",
    "rear_flow_change_amount_score",
    "front_flow_globality_score",
    "rear_flow_globality_score",
    "front_flow_transience_score",
    "rear_flow_transience_score",
    "front_flow_xy_impact_score",
    "rear_flow_xy_impact_score",
    "front_flow_high_change_cell_ratio",
    "rear_flow_high_change_cell_ratio",
    "front_flow_event_duration_steps",
    "rear_flow_event_duration_steps",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"TRAIN_DATA_DIR: {TRAIN_DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print("broad vibration columns:", BROAD_VIBRATION_COLUMNS)
print("broad vibration top_k:", BROAD_VIBRATION_TOP_K)
print("flow xy impact columns:", FLOW_XY_IMPACT_COLUMNS)
print("flow xy impact config:", FLOW_XY_IMPACT_CONFIG)

PROJECT_ROOT: /workspaces/forklift
TRAIN_DATA_DIR: /workspaces/forklift/data/movie_preprocess/normal
OUTPUT_DIR: /workspaces/forklift/outputs/anomaly_feature_poc
broad vibration columns: ['front_broad_vibration_score', 'rear_broad_vibration_score']
broad vibration top_k: 5
flow xy impact columns: ['front_flow_change_amount_score', 'front_flow_globality_score', 'front_flow_transience_score', 'front_flow_xy_impact_score', 'rear_flow_change_amount_score', 'rear_flow_globality_score', 'rear_flow_transience_score', 'rear_flow_xy_impact_score']
flow xy impact config: {'grid': (3, 3), 'min_valid_cell_ratio': 0.01, 'score_lower_quantile': 0.5, 'score_upper_quantile': 0.99, 'cell_score_upper_quantile': 0.95, 'event_threshold': 0.65, 'duration_alpha': 1.0, 'globality_weight': 0.5, 'transience_weight': 0.3, 'change_amount_weight': 0.2, 'globality_ratio_weight': 0.85, 'globality_intensity_weight': 0.15}


## 3. 正常データの検出

前処理済みデータは `sample_id_front.mp4` / `sample_id_rear.mp4` / `sample_id_audio.wav` の3ファイルを1サンプルとして扱います。

In [4]:
def discover_movie_preprocess_samples(data_dir: Path, manifest_path: Path | None = None) -> pd.DataFrame:
    rows = []
    for env_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        by_id: dict[str, dict] = {}
        for path in sorted(env_dir.glob("*_*.*")):
            stem = path.stem
            if "_" not in stem:
                continue
            sample_id, kind = stem.rsplit("_", 1)
            if kind not in {"front", "rear", "audio"}:
                continue
            row = by_id.setdefault(sample_id, {"sample_id": sample_id, "environment": env_dir.name})
            row[f"{kind}_path"] = path
        rows.extend(by_id.values())

    samples = pd.DataFrame(rows)
    if samples.empty:
        return samples

    for col in ["front_path", "rear_path", "audio_path"]:
        if col not in samples.columns:
            samples[col] = pd.NA

    if manifest_path and manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
        manifest = manifest[manifest["category"].eq("normal")].copy()
        manifest["sample_id"] = manifest["input_video_path"].astype(str).str.extract(r"(\d+)")
        keep_cols = [
            "sample_id", "input_duration_sec", "written_frame_pairs", "trim_start_sec", "trim_duration_sec",
            "status", "audio_status", "output_width", "output_height",
        ]
        manifest = manifest[[c for c in keep_cols if c in manifest.columns]].drop_duplicates("sample_id")
        samples = samples.merge(manifest, on="sample_id", how="left")

    samples["has_front"] = samples["front_path"].map(lambda p: isinstance(p, Path) and p.exists())
    samples["has_rear"] = samples["rear_path"].map(lambda p: isinstance(p, Path) and p.exists())
    samples["has_audio"] = samples["audio_path"].map(lambda p: isinstance(p, Path) and p.exists())
    samples["usable"] = samples[["has_front", "has_rear", "has_audio"]].all(axis=1)
    return samples.sort_values(["environment", "sample_id"]).reset_index(drop=True)

samples_df = discover_movie_preprocess_samples(TRAIN_DATA_DIR, MANIFEST_PATH)
print(samples_df.shape)
display(samples_df.head(10))
print(samples_df[["environment", "usable"]].value_counts(dropna=False))

(101, 17)


,sample_id,environment,audio_path,front_path,rear_path,input_duration_sec,written_frame_pairs,trim_start_sec,trim_duration_sec,status,audio_status,output_width,output_height,has_front,has_rear,has_audio,usable
0,1019,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.600000,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
1,1020,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.633333,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
2,1021,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.600000,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
3,1022,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.566667,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
4,1023,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.600000,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
5,1024,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.700000,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
6,1025,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.800000,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
7,1026,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.766667,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
8,1027,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.500000,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True
9,1028,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,15.733333,NaN,NaN,NaN,skipped_existing_preprocess_file,not_started,NaN,NaN,True,True,True,True


environment  usable
indoor       True      83
outdoor      True      18
Name: count, dtype: int64


## 4. 動画読み込みと前後カメラ分割ユーティリティ

今回の学習データは前後が分離済みですが、元の縦結合動画を確認するときにも使えるように `split_front_rear` と `extract_video_frames` は残しています。

In [5]:
def split_front_rear(frame: np.ndarray, front_is_top: bool = True) -> tuple[np.ndarray, np.ndarray]:
    h, _ = frame.shape[:2]
    top = frame[: h // 2]
    bottom = frame[h // 2 :]
    return (top, bottom) if front_is_top else (bottom, top)


def resize_keep_aspect(frame: np.ndarray, width: int | None) -> np.ndarray:
    if width is None or frame.shape[1] <= width:
        return frame
    scale = width / frame.shape[1]
    height = max(1, int(round(frame.shape[0] * scale)))
    return cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)


def extract_video_frames(
    video_path: str | Path,
    fps_sample: float = FPS_SAMPLE,
    resize_width: int | None = FRAME_RESIZE_WIDTH,
    split: bool = False,
    front_is_top: bool = True,
    max_duration_sec: float | None = None,
) -> list[dict] | tuple[list[dict], list[dict]]:
    video_path = Path(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        warnings.warn(f"Could not open video: {video_path}")
        return ([], []) if split else []

    src_fps = cap.get(cv2.CAP_PROP_FPS)
    if not src_fps or np.isnan(src_fps) or src_fps <= 0:
        src_fps = fps_sample
    frame_interval = max(1, int(round(src_fps / fps_sample)))

    frames = []
    frames_front, frames_rear = [], []
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        time_sec = frame_idx / src_fps
        if max_duration_sec is not None and time_sec > max_duration_sec:
            break
        if frame_idx % frame_interval == 0:
            if split:
                front, rear = split_front_rear(frame, front_is_top=front_is_top)
                frames_front.append({"time": time_sec, "frame": resize_keep_aspect(front, resize_width)})
                frames_rear.append({"time": time_sec, "frame": resize_keep_aspect(rear, resize_width)})
            else:
                frames.append({"time": time_sec, "frame": resize_keep_aspect(frame, resize_width)})
        frame_idx += 1
    cap.release()

    return (frames_front, frames_rear) if split else frames

## 5. 音声特徴抽出

`*_audio.wav` を優先して読み込みます。元動画しかない場合のために `extract_audio_from_video` も用意しています。

In [6]:
def extract_audio_from_video(video_path: str | Path, output_wav_path: str | Path, sr: int = AUDIO_SR) -> Path | None:
    video_path = Path(video_path)
    output_wav_path = Path(output_wav_path)
    if not shutil.which("ffmpeg"):
        warnings.warn("ffmpeg was not found. Use pre-extracted *_audio.wav or install ffmpeg.")
        return None
    cmd = [
        "ffmpeg", "-y", "-i", str(video_path), "-vn", "-ac", "1", "-ar", str(sr),
        "-hide_banner", "-loglevel", "error", str(output_wav_path),
    ]
    try:
        subprocess.run(cmd, check=True)
        return output_wav_path
    except subprocess.CalledProcessError as exc:
        warnings.warn(f"Audio extraction failed: {exc}")
        return None


def load_audio_mono(wav_path: str | Path, sr: int = AUDIO_SR) -> tuple[np.ndarray, int]:
    wav_path = Path(wav_path)
    if not wav_path.exists():
        return np.array([], dtype=np.float32), sr
    if librosa is not None:
        y, used_sr = librosa.load(wav_path, sr=sr, mono=True)
        return y.astype(np.float32), used_sr

    used_sr, y = wavfile.read(wav_path)
    if y.ndim > 1:
        y = y.mean(axis=1)
    y = y.astype(np.float32)
    max_abs = np.max(np.abs(y)) if y.size else 1.0
    if max_abs > 0:
        y = y / max_abs
    if used_sr != sr and y.size:
        gcd = math.gcd(used_sr, sr)
        y = resample_poly(y, sr // gcd, used_sr // gcd).astype(np.float32)
        used_sr = sr
    return y, used_sr


def extract_audio_features(
    wav_path: str | Path,
    window_sec: float = WINDOW_SEC,
    sr: int = AUDIO_SR,
    n_mels: int = N_MELS,
    video_id: str | None = None,
) -> pd.DataFrame:
    y, used_sr = load_audio_mono(wav_path, sr=sr)
    win = max(1, int(round(window_sec * used_sr)))
    if y.size == 0:
        return pd.DataFrame(columns=["video_id", "time_bin", "time", "audio_missing"])

    n_windows = int(math.ceil(len(y) / win))
    rows = []
    for i in range(n_windows):
        start = i * win
        end = min(len(y), start + win)
        segment = y[start:end]
        if segment.size == 0:
            continue
        spectrum = np.abs(np.fft.rfft(segment * np.hanning(segment.size)))
        freqs = np.fft.rfftfreq(segment.size, d=1.0 / used_sr)
        power = spectrum ** 2
        power_sum = float(power.sum()) + 1e-12
        centroid = float((freqs * power).sum() / power_sum)
        bandwidth = float(np.sqrt((((freqs - centroid) ** 2) * power).sum() / power_sum))
        high_freq_energy = float(power[freqs >= 3000].sum() / power_sum)

        row = {
            "video_id": video_id,
            "time_bin": i,
            "time": i * window_sec,
            "audio_rms": float(np.sqrt(np.mean(segment ** 2))),
            "audio_energy": float(np.mean(segment ** 2)),
            "audio_peak": float(np.max(np.abs(segment))),
            "audio_ptp": float(np.ptp(segment)),
            "audio_zcr": float(np.mean(np.abs(np.diff(np.signbit(segment))).astype(float))) if segment.size > 1 else 0.0,
            "audio_centroid": centroid,
            "audio_bandwidth": bandwidth,
            "audio_high_freq_energy": high_freq_energy,
            "audio_missing": 0,
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    if librosa is not None and len(df):
        mel = librosa.feature.melspectrogram(
            y=y, sr=used_sr, n_mels=n_mels, n_fft=min(2048, max(256, win)),
            hop_length=win, power=2.0,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max).T
        for m in range(n_mels):
            values = np.full(len(df), np.nan, dtype=float)
            upto = min(len(values), log_mel.shape[0])
            values[:upto] = log_mel[:upto, m]
            df[f"audio_mel_{m:02d}"] = values
    else:
        for m in range(n_mels):
            df[f"audio_mel_{m:02d}"] = 0.0

    return df

## 6. オプティカルフロー特徴抽出

Farneback 法でフローを計算し、画面全体と 3x3 グリッドの統計量に落とします。

In [7]:
def build_motion_valid_mask(frame_shape: tuple[int, ...], prefix: str | None = None) -> np.ndarray:
    """Return an all-true mask. Region masking is intentionally disabled."""
    h, w = int(frame_shape[0]), int(frame_shape[1])
    return np.ones((h, w), dtype=bool)


def build_feature_track_mask(frame_shape: tuple[int, ...], prefix: str | None = None) -> np.ndarray:
    valid = build_motion_valid_mask(frame_shape, prefix)
    return (valid.astype(np.uint8) * 255)


def masked_values(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> np.ndarray:
    selected = values[valid_mask]
    if selected.size == 0 and fallback_to_all:
        selected = values.reshape(-1)
    return selected[np.isfinite(selected)]


def masked_mean(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.mean(selected)) if selected.size else 0.0


def masked_std(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.std(selected)) if selected.size else 0.0


def masked_max(values: np.ndarray, valid_mask: np.ndarray, fallback_to_all: bool = True) -> float:
    selected = masked_values(values, valid_mask, fallback_to_all=fallback_to_all)
    return float(np.max(selected)) if selected.size else 0.0


def points_in_valid_mask(points: np.ndarray, valid_mask: np.ndarray) -> np.ndarray:
    h, w = valid_mask.shape
    x = np.rint(points[:, 0]).astype(int)
    y = np.rint(points[:, 1]).astype(int)
    in_bounds = (0 <= x) & (x < w) & (0 <= y) & (y < h)
    keep = np.zeros(points.shape[0], dtype=bool)
    keep[in_bounds] = valid_mask[y[in_bounds], x[in_bounds]]
    return keep



def resize_gray_for_flow(gray: np.ndarray, scale: float = FLOW_ANALYSIS_SCALE) -> tuple[np.ndarray, float, float]:
    """Downscale the image used by Farneback flow and return x/y scale factors."""
    scale = float(scale)
    if scale <= 0.0 or scale >= 1.0:
        return gray, 1.0, 1.0
    h, w = gray.shape[:2]
    new_w = max(8, int(round(w * scale)))
    new_h = max(8, int(round(h * scale)))
    if new_w == w and new_h == h:
        return gray, 1.0, 1.0
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized, new_w / max(w, 1), new_h / max(h, 1)

def flow_xy_raw_output_columns(prefix: str, grid: tuple[int, int] = FLOW_GRID) -> list[str]:
    cols = [
        f"{prefix}_flow_change_amount_raw",
        f"{prefix}_flow_cell_change_mean_raw",
        f"{prefix}_flow_cell_change_max_raw",
        f"{prefix}_flow_cell_change_p95_raw",
        f"{prefix}_flow_valid_cell_count",
        f"{prefix}_flow_cell_count",
    ]
    for gy in range(grid[0]):
        for gx in range(grid[1]):
            cols.append(f"{prefix}_flow_cell_{gy}_{gx}_change_raw")
    return cols


def _empty_flow_row(prefix: str, time_sec: float, time_bin: int, video_id: str | None, flow_dt_sec: float | None = None) -> dict:
    default_dt = 1.0 / max(float(FPS_SAMPLE), 1e-6)
    row = {
        "video_id": video_id,
        "time_bin": time_bin,
        "time": time_sec,
        f"{prefix}_flow_dt_sec": float(flow_dt_sec if flow_dt_sec is not None else default_dt),
    }
    base_cols = [
        "flow_mag_mean", "flow_mag_std", "flow_mag_max", "flow_angle_mean", "flow_angle_std",
        "flow_x_mean", "flow_x_std", "flow_y_mean", "flow_y_std", "flow_failed",
    ]
    for c in base_cols:
        row[f"{prefix}_{c}"] = 1.0 if c == "flow_failed" else 0.0
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            for c in ["mag_mean", "mag_std", "x_mean", "y_mean", "valid_ratio"]:
                row[f"{prefix}_flow_cell_{gy}_{gx}_{c}"] = 0.0
    for col in flow_xy_raw_output_columns(prefix):
        row[col] = 0.0
    return row


def _diff_per_second(work: pd.DataFrame, col: str, dt: pd.Series) -> np.ndarray:
    delta = work.groupby("video_id", dropna=False)[col].diff().fillna(0.0).to_numpy(dtype=float)
    return delta / dt.to_numpy(dtype=float)


def _nanmean_or_zero(values: np.ndarray, axis: int = 1) -> np.ndarray:
    with np.errstate(invalid="ignore"):
        out = np.nanmean(values, axis=axis)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def _nanmax_or_zero(values: np.ndarray, axis: int = 1) -> np.ndarray:
    all_nan = np.isnan(values).all(axis=axis)
    safe = np.where(np.isnan(values), -np.inf, values)
    out = np.max(safe, axis=axis)
    out[all_nan] = 0.0
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def _nanpercentile_or_zero(values: np.ndarray, percentile: float, axis: int = 1) -> np.ndarray:
    out = np.zeros(values.shape[0], dtype=float)
    for i, row in enumerate(values):
        finite = row[np.isfinite(row)]
        if finite.size:
            out[i] = float(np.percentile(finite, percentile))
    return out


def add_flow_xy_change_window_features(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Add raw x/y flow-difference features. Normal-data calibration is applied later."""
    if df.empty:
        return df
    x_col = f"{prefix}_flow_x_mean"
    y_col = f"{prefix}_flow_y_mean"
    if x_col not in df.columns or y_col not in df.columns:
        return df.drop(columns=[f"{prefix}_flow_dt_sec"], errors="ignore")

    work = df.sort_values(["video_id", "time"]).reset_index(drop=True).copy()
    default_dt = 1.0 / max(float(FPS_SAMPLE), 1e-6)
    dt_col = f"{prefix}_flow_dt_sec"
    if dt_col in work.columns:
        dt = work[dt_col].replace([np.inf, -np.inf], np.nan).fillna(default_dt).clip(lower=1e-6)
    else:
        dt = work.groupby("video_id", dropna=False)["time"].diff().fillna(default_dt).clip(lower=1e-6)

    dx = _diff_per_second(work, x_col, dt)
    dy = _diff_per_second(work, y_col, dt)
    amount_col = f"{prefix}_flow_change_amount_raw"
    mean_col = f"{prefix}_flow_cell_change_mean_raw"
    max_col = f"{prefix}_flow_cell_change_max_raw"
    p95_col = f"{prefix}_flow_cell_change_p95_raw"
    valid_count_col = f"{prefix}_flow_valid_cell_count"
    cell_count_col = f"{prefix}_flow_cell_count"
    work[amount_col] = np.hypot(dx, dy)

    cell_change_cols = []
    cell_changes = []
    valid_masks = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            cx = f"{prefix}_flow_cell_{gy}_{gx}_x_mean"
            cy = f"{prefix}_flow_cell_{gy}_{gx}_y_mean"
            valid_col = f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio"
            change_col = f"{prefix}_flow_cell_{gy}_{gx}_change_raw"
            if cx not in work.columns or cy not in work.columns:
                continue
            cdx = _diff_per_second(work, cx, dt)
            cdy = _diff_per_second(work, cy, dt)
            change = np.hypot(cdx, cdy)
            if valid_col in work.columns:
                valid = work[valid_col].to_numpy(dtype=float) >= float(FLOW_MIN_VALID_CELL_RATIO)
            else:
                valid = np.ones(len(work), dtype=bool)
            change = np.where(valid, change, np.nan)
            work[change_col] = change
            cell_change_cols.append(change_col)
            cell_changes.append(change)
            valid_masks.append(valid)

    if cell_changes:
        change_matrix = np.vstack(cell_changes).T
        valid_matrix = np.vstack(valid_masks).T
        work[mean_col] = _nanmean_or_zero(change_matrix, axis=1)
        work[max_col] = _nanmax_or_zero(change_matrix, axis=1)
        work[p95_col] = _nanpercentile_or_zero(change_matrix, 95, axis=1)
        work[valid_count_col] = valid_matrix.sum(axis=1).astype(float)
        work[cell_count_col] = float(len(cell_change_cols))
    else:
        work[mean_col] = 0.0
        work[max_col] = 0.0
        work[p95_col] = 0.0
        work[valid_count_col] = 0.0
        work[cell_count_col] = 0.0

    agg_spec = {
        amount_col: (amount_col, "max"),
        mean_col: (mean_col, "max"),
        max_col: (max_col, "max"),
        p95_col: (p95_col, "max"),
        valid_count_col: (valid_count_col, "max"),
        cell_count_col: (cell_count_col, "max"),
    }
    for col in cell_change_cols:
        agg_spec[col] = (col, "max")

    window_features = work.groupby(["video_id", "time_bin"], as_index=False).agg(**agg_spec)
    out = df.merge(window_features, on=["video_id", "time_bin"], how="left")
    for col in flow_xy_raw_output_columns(prefix):
        if col not in out.columns:
            out[col] = 0.0
        else:
            out[col] = out[col].fillna(0.0)
    return out.drop(columns=[f"{prefix}_flow_dt_sec"], errors="ignore")


def compute_optical_flow_features(
    frames: list[dict],
    prefix: str,
    window_sec: float = WINDOW_SEC,
    grid: tuple[int, int] = FLOW_GRID,
    video_id: str | None = None,
) -> pd.DataFrame:
    if len(frames) < 2:
        return pd.DataFrame()

    rows = []
    prev_gray_original = cv2.cvtColor(frames[0]["frame"], cv2.COLOR_BGR2GRAY)
    prev_gray, prev_scale_x, prev_scale_y = resize_gray_for_flow(prev_gray_original)
    prev_time = float(frames[0]["time"])
    for item in frames[1:]:
        time_sec = float(item["time"])
        flow_dt_sec = max(time_sec - prev_time, 1e-6)
        time_bin = int(round(time_sec / window_sec))
        gray_original = cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY)
        gray, scale_x, scale_y = resize_gray_for_flow(gray_original)
        try:
            flow = cv2.calcOpticalFlowFarneback(
                prev_gray, gray, None,
                pyr_scale=0.5, levels=3, winsize=15, iterations=3,
                poly_n=5, poly_sigma=1.2, flags=0,
            )
            # Farneback returns displacement in the downscaled image coordinate.
            # Convert it back to the resized-frame coordinate so thresholds stay comparable.
            fx = flow[..., 0] / max(scale_x, 1e-6)
            fy = flow[..., 1] / max(scale_y, 1e-6)
            mag, angle = cv2.cartToPolar(fx, fy, angleInDegrees=False)
            valid_mask = build_motion_valid_mask(mag.shape, prefix)
            row = {
                "video_id": video_id,
                "time_bin": time_bin,
                "time": time_sec,
                f"{prefix}_flow_dt_sec": flow_dt_sec,
                f"{prefix}_flow_mag_mean": masked_mean(mag, valid_mask),
                f"{prefix}_flow_mag_std": masked_std(mag, valid_mask),
                f"{prefix}_flow_mag_max": masked_max(mag, valid_mask),
                f"{prefix}_flow_angle_mean": masked_mean(angle, valid_mask),
                f"{prefix}_flow_angle_std": masked_std(angle, valid_mask),
                f"{prefix}_flow_x_mean": masked_mean(fx, valid_mask),
                f"{prefix}_flow_x_std": masked_std(fx, valid_mask),
                f"{prefix}_flow_y_mean": masked_mean(fy, valid_mask),
                f"{prefix}_flow_y_std": masked_std(fy, valid_mask),
                f"{prefix}_flow_failed": 0,
            }
            h, w = mag.shape
            gy_n, gx_n = grid
            for gy in range(gy_n):
                y0, y1 = int(h * gy / gy_n), int(h * (gy + 1) / gy_n)
                for gx in range(gx_n):
                    x0, x1 = int(w * gx / gx_n), int(w * (gx + 1) / gx_n)
                    cell_mag = mag[y0:y1, x0:x1]
                    cell_fx = fx[y0:y1, x0:x1]
                    cell_fy = fy[y0:y1, x0:x1]
                    cell_mask = valid_mask[y0:y1, x0:x1]
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_mean"] = masked_mean(cell_mag, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_mag_std"] = masked_std(cell_mag, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_x_mean"] = masked_mean(cell_fx, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_y_mean"] = masked_mean(cell_fy, cell_mask, fallback_to_all=False)
                    row[f"{prefix}_flow_cell_{gy}_{gx}_valid_ratio"] = float(np.mean(cell_mask)) if cell_mask.size else 0.0
        except Exception as exc:
            warnings.warn(f"{prefix} optical flow failed at {time_sec:.2f}s: {exc}")
            row = _empty_flow_row(prefix, time_sec, time_bin, video_id, flow_dt_sec=flow_dt_sec)
        rows.append(row)
        prev_gray = gray
        prev_scale_x = scale_x
        prev_scale_y = scale_y
        prev_time = time_sec

    return add_flow_xy_change_window_features(pd.DataFrame(rows), prefix)

## 7. グローバルモーション特徴抽出

特徴点追跡と部分アフィン推定で、カメラ全体の揺れ・回転・推定失敗を特徴量化します。

In [8]:
def _global_failure_row(prefix: str, time_sec: float, time_bin: int, video_id: str | None, num_points: int = 0) -> dict:
    return {
        "video_id": video_id,
        "time_bin": time_bin,
        "time": time_sec,
        f"{prefix}_global_dx": 0.0,
        f"{prefix}_global_dy": 0.0,
        f"{prefix}_global_translation_norm": 0.0,
        f"{prefix}_global_rotation": 0.0,
        f"{prefix}_global_scale": 1.0,
        f"{prefix}_global_ransac_residual": 0.0,
        f"{prefix}_global_inlier_ratio": 0.0,
        f"{prefix}_global_num_tracked_points": float(num_points),
        f"{prefix}_global_failed": 1,
    }


def compute_global_motion_features(
    frames: list[dict],
    prefix: str,
    window_sec: float = WINDOW_SEC,
    max_corners: int = MAX_CORNERS,
    video_id: str | None = None,
) -> pd.DataFrame:
    if len(frames) < 2:
        return pd.DataFrame()

    lk_params = dict(winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    feature_params = dict(maxCorners=max_corners, qualityLevel=0.01, minDistance=7, blockSize=7)

    def detect_motion_points(gray_image: np.ndarray) -> np.ndarray | None:
        return cv2.goodFeaturesToTrack(gray_image, mask=build_feature_track_mask(gray_image.shape, prefix), **feature_params)

    rows = []
    prev_gray = cv2.cvtColor(frames[0]["frame"], cv2.COLOR_BGR2GRAY)
    prev_valid_mask = build_motion_valid_mask(prev_gray.shape, prefix)
    prev_pts = detect_motion_points(prev_gray)

    for item in frames[1:]:
        time_sec = float(item["time"])
        time_bin = int(round(time_sec / window_sec))
        gray = cv2.cvtColor(item["frame"], cv2.COLOR_BGR2GRAY)
        current_valid_mask = build_motion_valid_mask(gray.shape, prefix)

        if prev_pts is None or len(prev_pts) < 6:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, 0 if prev_pts is None else len(prev_pts)))
            prev_gray = gray
            prev_valid_mask = current_valid_mask
            prev_pts = detect_motion_points(prev_gray)
            continue

        next_pts, status, _err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, **lk_params)
        if next_pts is None or status is None:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, len(prev_pts)))
            prev_gray = gray
            prev_valid_mask = current_valid_mask
            prev_pts = detect_motion_points(prev_gray)
            continue

        good_prev = prev_pts[status.ravel() == 1].reshape(-1, 2)
        good_next = next_pts[status.ravel() == 1].reshape(-1, 2)
        track_mask = points_in_valid_mask(good_prev, prev_valid_mask) & points_in_valid_mask(good_next, current_valid_mask)
        good_prev = good_prev[track_mask]
        good_next = good_next[track_mask]
        if len(good_prev) < 6:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, len(good_prev)))
            prev_gray = gray
            prev_valid_mask = current_valid_mask
            prev_pts = detect_motion_points(prev_gray)
            continue

        M, inliers = cv2.estimateAffinePartial2D(good_prev, good_next, method=cv2.RANSAC, ransacReprojThreshold=3.0)
        if M is None or inliers is None:
            rows.append(_global_failure_row(prefix, time_sec, time_bin, video_id, len(good_prev)))
            prev_gray = gray
            prev_valid_mask = current_valid_mask
            prev_pts = detect_motion_points(prev_gray)
            continue

        a, b, dx = M[0]
        c, d, dy = M[1]
        scale = float(np.sqrt(a * a + c * c))
        rotation = float(np.arctan2(c, a))
        translation_norm = float(np.sqrt(dx * dx + dy * dy))
        inlier_mask = inliers.ravel().astype(bool)
        pred = cv2.transform(good_prev.reshape(-1, 1, 2), M).reshape(-1, 2)
        residuals = np.linalg.norm(pred - good_next, axis=1)
        residual = float(np.mean(residuals[inlier_mask])) if inlier_mask.any() else float(np.mean(residuals))
        inlier_ratio = float(inlier_mask.mean())

        rows.append({
            "video_id": video_id,
            "time_bin": time_bin,
            "time": time_sec,
            f"{prefix}_global_dx": float(dx),
            f"{prefix}_global_dy": float(dy),
            f"{prefix}_global_translation_norm": translation_norm,
            f"{prefix}_global_rotation": rotation,
            f"{prefix}_global_scale": scale,
            f"{prefix}_global_ransac_residual": residual,
            f"{prefix}_global_inlier_ratio": inlier_ratio,
            f"{prefix}_global_num_tracked_points": float(len(good_prev)),
            f"{prefix}_global_failed": 0,
        })

        prev_gray = gray
        prev_valid_mask = current_valid_mask
        prev_pts = detect_motion_points(prev_gray)

    return pd.DataFrame(rows)

## 8. センサー特徴の取り込み オプション

センサー CSV がない場合は `sensor_missing=1` のみを付け、映像・音声特徴だけで進めます。

In [9]:
def classify_motion_state(speed: float, steering: float, steering_threshold: float = 5.0, low_speed_threshold: float = 0.5) -> str:
    if abs(speed) < 0.1:
        return "stop"
    if abs(steering) > steering_threshold:
        return "turning"
    if abs(speed) < low_speed_threshold:
        return "low_speed"
    return "straight"


def load_sensor_features(sensor_csv_path: str | Path | None, window_sec: float = WINDOW_SEC, video_id: str | None = None) -> pd.DataFrame:
    if sensor_csv_path is None or not Path(sensor_csv_path).exists():
        return pd.DataFrame({"video_id": [video_id], "time_bin": [0], "time": [0.0], "sensor_missing": [1]})

    df = pd.read_csv(sensor_csv_path)
    if "time" not in df.columns:
        raise ValueError("sensor csv must have a 'time' column")
    df["time_bin"] = np.round(df["time"] / window_sec).astype(int)
    df["video_id"] = video_id

    for col in ["speed", "steering", "accel", "throttle", "brake"]:
        if col not in df.columns:
            df[col] = 0.0

    df["speed_abs"] = df["speed"].abs()
    df["steering_abs"] = df["steering"].abs()
    df["stop_flag"] = (df["speed_abs"] < 0.1).astype(int)
    df["low_speed_flag"] = (df["speed_abs"] < 0.5).astype(int)
    df["turning_flag"] = (df["steering_abs"] > 5.0).astype(int)
    df["moving_state"] = [classify_motion_state(s, st) for s, st in zip(df["speed"], df["steering"])]
    df["sensor_missing"] = 0

    numeric = ["speed", "speed_abs", "steering", "steering_abs", "accel", "throttle", "brake", "stop_flag", "low_speed_flag", "turning_flag", "sensor_missing"]
    grouped = df.groupby(["video_id", "time_bin"], as_index=False)[numeric].mean()
    state = df.groupby(["video_id", "time_bin"])["moving_state"].agg(lambda x: x.mode().iat[0] if len(x.mode()) else "unknown").reset_index()
    out = grouped.merge(state, on=["video_id", "time_bin"], how="left")
    out["time"] = out["time_bin"] * window_sec
    return out

## 9. 特徴量の時間結合

In [10]:
def _aggregate_feature_df(df: pd.DataFrame, window_sec: float = WINDOW_SEC) -> pd.DataFrame:
    """Ensure each feature source has one row per video_id/time_bin before merging."""
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    if "video_id" not in df.columns:
        df["video_id"] = "unknown"
    if "time_bin" not in df.columns:
        if "time" not in df.columns:
            raise ValueError("feature df must have either time_bin or time")
        df["time_bin"] = np.round(df["time"] / window_sec).astype(int)

    key_cols = ["video_id", "time_bin"]
    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    other_cols = [c for c in df.columns if c not in set(key_cols + numeric_cols + ["time"])]

    out = df.groupby(key_cols, as_index=False)[numeric_cols].mean() if numeric_cols else df[key_cols].drop_duplicates()
    for col in other_cols:
        values = df.groupby(key_cols)[col].agg(lambda x: x.dropna().mode().iat[0] if len(x.dropna().mode()) else "unknown").reset_index()
        out = out.merge(values, on=key_cols, how="left")

    out["time"] = out["time_bin"] * window_sec
    return out


def align_features_by_time(feature_dfs: Iterable[pd.DataFrame], window_sec: float = WINDOW_SEC) -> pd.DataFrame:
    valid = [_aggregate_feature_df(df, window_sec=window_sec) for df in feature_dfs if df is not None and len(df)]
    valid = [df for df in valid if len(df)]
    if not valid:
        return pd.DataFrame()

    merged = valid[0]
    key_cols = ["video_id", "time_bin"]
    for df in valid[1:]:
        merged = merged.merge(df.drop(columns=["time"], errors="ignore"), on=key_cols, how="outer")
    merged["time"] = merged["time_bin"] * window_sec
    merged = merged.sort_values(key_cols).reset_index(drop=True)

    numeric_cols = [c for c in merged.select_dtypes(include=[np.number]).columns if c not in {"time", "time_bin"}]
    merged[numeric_cols] = merged.groupby("video_id", group_keys=False)[numeric_cols].apply(lambda g: g.ffill().bfill())
    merged[numeric_cols] = merged[numeric_cols].fillna(0.0)
    for c in merged.select_dtypes(exclude=[np.number]).columns:
        if c != "video_id":
            merged[c] = merged[c].fillna("unknown")
    return merged


FLOW_XY_SCORE_SUFFIXES = (
    "flow_change_amount_score",
    "flow_globality_score",
    "flow_transience_score",
    "flow_xy_impact_score",
)
FLOW_XY_DIAGNOSTIC_SUFFIXES = (
    "flow_change_amount_raw",
    "flow_cell_change_mean_raw",
    "flow_cell_change_max_raw",
    "flow_cell_change_p95_raw",
    "flow_high_change_cell_count",
    "flow_valid_cell_count",
    "flow_high_change_cell_ratio",
    "flow_event_duration_steps",
    "flow_cell_count",
)


def flow_xy_score_columns(prefix: str) -> list[str]:
    return [f"{prefix}_{suffix}" for suffix in FLOW_XY_SCORE_SUFFIXES]


def flow_xy_diagnostic_columns(prefix: str) -> list[str]:
    return [f"{prefix}_{suffix}" for suffix in FLOW_XY_DIAGNOSTIC_SUFFIXES]


def flow_xy_cell_change_columns(df: pd.DataFrame, prefix: str) -> list[str]:
    start = f"{prefix}_flow_cell_"
    return sorted([c for c in df.columns if c.startswith(start) and c.endswith("_change_raw")])


def add_flow_change_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure new flow xy impact columns exist for front/rear streams."""
    df = df.copy()
    sort_cols = [c for c in ["video_id", "time_bin", "time"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    for prefix in ["front", "rear"]:
        for col in [*flow_xy_score_columns(prefix), *flow_xy_diagnostic_columns(prefix)]:
            if col not in df.columns:
                df[col] = 0.0
    return df


def add_cross_camera_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pairs = [
        ("flow_mag_mean", "front_flow_mag_mean", "rear_flow_mag_mean"),
        ("flow_xy_impact_score", "front_flow_xy_impact_score", "rear_flow_xy_impact_score"),
        ("flow_globality_score", "front_flow_globality_score", "rear_flow_globality_score"),
        ("flow_change_amount_score", "front_flow_change_amount_score", "rear_flow_change_amount_score"),
        ("global_translation_norm", "front_global_translation_norm", "rear_global_translation_norm"),
        ("global_rotation_abs", "front_global_rotation", "rear_global_rotation"),
    ]
    for name, front_col, rear_col in pairs:
        if front_col in df.columns and rear_col in df.columns:
            f = df[front_col].abs() if "rotation" in name else df[front_col]
            r = df[rear_col].abs() if "rotation" in name else df[rear_col]
            df[f"front_rear_{name}_mean"] = (f + r) / 2
            df[f"front_rear_{name}_diff_abs"] = (f - r).abs()
            df[f"front_rear_{name}_both_high"] = f * r
    return df


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_flow_change_features(df)
    df = add_cross_camera_features(df)
    return df


def _finite_values(values) -> np.ndarray:
    arr = np.asarray(values, dtype=float).reshape(-1)
    arr = arr[np.isfinite(arr)]
    return arr


def _fit_percentile_scale(values, lower_q: float, upper_q: float) -> dict[str, float]:
    finite = _finite_values(values)
    if finite.size == 0:
        return {"lower": 0.0, "upper": 1.0}
    lower = float(np.quantile(finite, lower_q))
    upper = float(np.quantile(finite, upper_q))
    if not np.isfinite(upper) or upper <= lower:
        upper = lower + max(abs(lower) * 0.1, 1e-6)
    return {"lower": lower, "upper": upper}


def _score_from_scale(values, scale: dict[str, float]) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    lower = float(scale.get("lower", 0.0))
    upper = float(scale.get("upper", lower + 1.0))
    denom = max(upper - lower, 1e-6)
    return np.clip((np.nan_to_num(arr, nan=lower, posinf=upper, neginf=lower) - lower) / denom, 0.0, 1.0)


def _fit_flow_xy_prefix_calibration(df: pd.DataFrame, prefix: str) -> dict:
    amount_col = f"{prefix}_flow_change_amount_raw"
    amount_scale = _fit_percentile_scale(
        df[amount_col].to_numpy(dtype=float) if amount_col in df.columns else [],
        FLOW_SCORE_LOWER_QUANTILE,
        FLOW_SCORE_UPPER_QUANTILE,
    )
    cell_cols = flow_xy_cell_change_columns(df, prefix)
    if cell_cols:
        cell_values = df[cell_cols].to_numpy(dtype=float).reshape(-1)
    else:
        cell_values = []
    cell_scale = _fit_percentile_scale(
        cell_values,
        FLOW_SCORE_LOWER_QUANTILE,
        FLOW_CELL_SCORE_UPPER_QUANTILE,
    )
    return {
        "amount_scale": amount_scale,
        "cell_scale": cell_scale,
        "cell_columns": cell_cols,
    }


def fit_flow_xy_impact_calibration(df: pd.DataFrame, prefixes: tuple[str, ...] = ("front", "rear")) -> dict:
    return {
        "config": dict(FLOW_XY_IMPACT_CONFIG),
        "prefix_stats": {prefix: _fit_flow_xy_prefix_calibration(df, prefix) for prefix in prefixes},
    }


def _event_duration_steps(active: np.ndarray) -> np.ndarray:
    active = np.asarray(active, dtype=bool)
    durations = np.zeros(active.size, dtype=float)
    start = None
    for i, flag in enumerate(active):
        if flag and start is None:
            start = i
        ended = (not flag or i == active.size - 1) and start is not None
        if ended:
            end = i if not flag else i + 1
            durations[start:end] = float(end - start)
            start = None
    return durations


def _duration_by_video(df: pd.DataFrame, active: np.ndarray) -> pd.Series:
    durations = pd.Series(0.0, index=df.index, name="event_duration_steps")
    if len(df) == 0:
        return durations
    active_series = pd.Series(active, index=df.index)
    sort_cols = [c for c in ["video_id", "time_bin", "time"] if c in df.columns]
    ordered = df.sort_values(sort_cols) if sort_cols else df
    group_key = "video_id" if "video_id" in ordered.columns else None
    groups = ordered.groupby(group_key, sort=False, dropna=False) if group_key else [(None, ordered)]
    for _, group in groups:
        group_active = active_series.loc[group.index].to_numpy(dtype=bool)
        durations.loc[group.index] = _event_duration_steps(group_active)
    return durations


def apply_flow_xy_impact_calibration(
    df: pd.DataFrame,
    calibration: dict,
    prefixes: tuple[str, ...] = ("front", "rear"),
) -> pd.DataFrame:
    result = add_flow_change_features(df)
    prefix_stats = calibration.get("prefix_stats", {}) if isinstance(calibration, dict) else {}
    weight_sum = max(FLOW_GLOBALITY_WEIGHT + FLOW_TRANSIENCE_WEIGHT + FLOW_CHANGE_AMOUNT_WEIGHT, 1e-6)

    for prefix in prefixes:
        stats = prefix_stats.get(prefix, {})
        amount_col = f"{prefix}_flow_change_amount_raw"
        amount_score_col = f"{prefix}_flow_change_amount_score"
        globality_col = f"{prefix}_flow_globality_score"
        transience_col = f"{prefix}_flow_transience_score"
        impact_col = f"{prefix}_flow_xy_impact_score"
        high_count_col = f"{prefix}_flow_high_change_cell_count"
        valid_count_col = f"{prefix}_flow_valid_cell_count"
        ratio_col = f"{prefix}_flow_high_change_cell_ratio"
        duration_col = f"{prefix}_flow_event_duration_steps"
        cell_count_col = f"{prefix}_flow_cell_count"

        if not stats or amount_col not in result.columns:
            for col in [amount_score_col, globality_col, transience_col, impact_col, high_count_col, ratio_col, duration_col]:
                result[col] = 0.0
            continue

        amount_score = _score_from_scale(result[amount_col].to_numpy(dtype=float), stats.get("amount_scale", {}))
        result[amount_score_col] = amount_score

        cell_cols = [c for c in stats.get("cell_columns", []) if c in result.columns]
        if cell_cols:
            cell_raw = result[cell_cols].to_numpy(dtype=float)
            cell_scores = _score_from_scale(cell_raw, stats.get("cell_scale", {}))
            valid_matrix = np.isfinite(cell_raw)
            raw_valid_counts = (
                result[valid_count_col].to_numpy(dtype=float)
                if valid_count_col in result.columns
                else valid_matrix.sum(axis=1).astype(float)
            )
            valid_counts = np.minimum(np.nan_to_num(raw_valid_counts, nan=0.0), valid_matrix.sum(axis=1).astype(float))
            high_matrix = (cell_scores >= float(FLOW_EVENT_THRESHOLD)) & valid_matrix
            high_counts = np.minimum(high_matrix.sum(axis=1).astype(float), valid_counts)
            cell_mean_score = np.divide(
                (cell_scores * valid_matrix).sum(axis=1),
                valid_counts,
                out=np.zeros(len(result), dtype=float),
                where=valid_counts > 0,
            )
        else:
            valid_counts = np.zeros(len(result), dtype=float)
            high_counts = np.zeros(len(result), dtype=float)
            cell_mean_score = np.zeros(len(result), dtype=float)

        high_ratio = np.divide(high_counts, valid_counts, out=np.zeros(len(result), dtype=float), where=valid_counts > 0)
        globality = np.clip(
            float(FLOW_GLOBALITY_RATIO_WEIGHT) * high_ratio
            + float(FLOW_GLOBALITY_INTENSITY_WEIGHT) * np.sqrt(np.clip(high_ratio * cell_mean_score, 0.0, 1.0)),
            0.0,
            1.0,
        )
        event_signal = np.maximum(amount_score, globality)
        active = event_signal >= float(FLOW_EVENT_THRESHOLD)
        durations = _duration_by_video(result, active)
        duration_values = durations.to_numpy(dtype=float)
        transience = np.zeros(len(result), dtype=float)
        has_duration = duration_values > 0
        transience[has_duration] = 1.0 / np.power(duration_values[has_duration], float(FLOW_DURATION_ALPHA))
        impact = np.clip(
            (
                float(FLOW_GLOBALITY_WEIGHT) * globality
                + float(FLOW_TRANSIENCE_WEIGHT) * transience
                + float(FLOW_CHANGE_AMOUNT_WEIGHT) * amount_score
            ) / weight_sum,
            0.0,
            1.0,
        )

        result[globality_col] = globality
        result[transience_col] = transience
        result[impact_col] = impact
        result[high_count_col] = high_counts
        result[valid_count_col] = valid_counts
        result[ratio_col] = high_ratio
        result[duration_col] = durations.to_numpy(dtype=float)
        if cell_count_col not in result.columns:
            result[cell_count_col] = valid_counts
    return result


def robust_positive_zscore(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.zeros((0,), dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return np.zeros_like(values, dtype=float)
    median = float(np.median(finite))
    mad = float(np.median(np.abs(finite - median)))
    scale = max(1.4826 * mad, 1e-6)
    return np.maximum((np.nan_to_num(values, nan=median) - median) / scale, 0.0)


def percentile_normalize_0_1(
    values: np.ndarray | pd.Series,
    lower_percentile: float = DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
    upper_percentile: float = DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
    positive_only: bool = True,
) -> np.ndarray:
    values_arr = np.asarray(values, dtype=float)
    out = np.zeros_like(values_arr, dtype=float)
    finite_mask = np.isfinite(values_arr)
    if not finite_mask.any():
        return out
    fit_values = values_arr[finite_mask]
    if positive_only:
        fit_values = fit_values[fit_values > FLOW_SCORE_MIN_VISIBLE]
    if fit_values.size == 0:
        return out
    lower_q = float(np.clip(float(lower_percentile), 0.0, 100.0))
    upper_q = float(np.clip(float(upper_percentile), 0.0, 100.0))
    if upper_q < lower_q:
        lower_q, upper_q = upper_q, lower_q
    lower = float(np.percentile(fit_values, lower_q))
    upper = float(np.percentile(fit_values, upper_q))
    if upper <= lower:
        out[finite_mask] = np.where(values_arr[finite_mask] > FLOW_SCORE_MIN_VISIBLE, 1.0, 0.0)
    else:
        out[finite_mask] = np.clip((values_arr[finite_mask] - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    if positive_only:
        out = np.where(values_arr > FLOW_SCORE_MIN_VISIBLE, out, 0.0)
    return out.astype(float)


def build_flow_window_starts(duration_sec: float, window_sec: float = FLOW_SCORE_WINDOW_SEC, hop_sec: float = FLOW_SCORE_HOP_SEC) -> list[float]:
    max_start = max(float(duration_sec) - float(window_sec), 0.0)
    starts = np.arange(0.0, max_start + 1e-9, float(hop_sec), dtype=np.float32).tolist()
    return [float(value) for value in (starts if starts else [0.0])]


def compute_flow_direction_change_abs(raw_flow_df: pd.DataFrame, x_col: str, y_col: str) -> pd.Series:
    """Compute wrapped flow direction change in radians, ignoring near-zero vectors."""
    if x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.Series(dtype=float)
    ordered = raw_flow_df.sort_values("time")
    x = ordered[x_col].astype(float).to_numpy()
    y = ordered[y_col].astype(float).to_numpy()
    magnitude = np.hypot(x, y)
    angle = np.arctan2(y, x)
    delta = np.diff(angle, prepend=angle[:1])
    wrapped_delta = np.arctan2(np.sin(delta), np.cos(delta))
    direction_change = np.abs(wrapped_delta)
    valid = (magnitude >= FLOW_DIRECTION_MIN_MAG) & (np.r_[magnitude[:1], magnitude[:-1]] >= FLOW_DIRECTION_MIN_MAG)
    direction_change = np.where(valid, direction_change, 0.0)
    direction_change = np.clip(direction_change, 0.0, np.pi)
    return pd.Series(direction_change, index=ordered.index).reindex(raw_flow_df.index).fillna(0.0)


def resolve_direction_jitter_thresholds(
    scores: np.ndarray | pd.Series,
    high_threshold: float = DIRECTION_JITTER_HIGH_THRESHOLD,
    low_threshold: float = DIRECTION_JITTER_LOW_THRESHOLD,
    threshold_mode: str = DIRECTION_JITTER_THRESHOLD_MODE,
    high_percentile: float = DIRECTION_JITTER_HIGH_PERCENTILE,
    low_percentile: float = DIRECTION_JITTER_LOW_PERCENTILE,
) -> tuple[float, float]:
    finite_scores = np.asarray(scores, dtype=float)
    positive_scores = finite_scores[np.isfinite(finite_scores) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)]
    if str(threshold_mode).lower() == "percentile" and positive_scores.size:
        high = float(np.percentile(positive_scores, np.clip(float(high_percentile), 0.0, 100.0)))
        low = float(np.percentile(positive_scores, np.clip(float(low_percentile), 0.0, 100.0)))
    else:
        high = float(high_threshold)
        low = float(low_threshold)
    if low > high:
        low = high
    return high, low


def mark_direction_jitter_hysteresis_windows(
    window_df: pd.DataFrame,
    high_threshold: float = DIRECTION_JITTER_HIGH_THRESHOLD,
    low_threshold: float = DIRECTION_JITTER_LOW_THRESHOLD,
    threshold_mode: str = DIRECTION_JITTER_THRESHOLD_MODE,
    high_percentile: float = DIRECTION_JITTER_HIGH_PERCENTILE,
    low_percentile: float = DIRECTION_JITTER_LOW_PERCENTILE,
) -> pd.DataFrame:
    """Select high-score windows and adjacent low-threshold windows within one grid."""
    window_df = window_df.sort_values("window_start_sec").copy()
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    finite_scores = np.nan_to_num(scores, nan=-np.inf)
    resolved_high_threshold, resolved_low_threshold = resolve_direction_jitter_thresholds(
        scores,
        high_threshold=high_threshold,
        low_threshold=low_threshold,
        threshold_mode=threshold_mode,
        high_percentile=high_percentile,
        low_percentile=low_percentile,
    )
    seed_mask = (finite_scores >= resolved_high_threshold) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)
    low_mask = (finite_scores >= resolved_low_threshold) & (finite_scores > FLOW_SCORE_MIN_VISIBLE)
    selected = np.zeros(len(window_df), dtype=bool)

    for seed_index in np.flatnonzero(seed_mask):
        selected[seed_index] = True
        left = seed_index - 1
        while left >= 0 and low_mask[left]:
            selected[left] = True
            left -= 1
        right = seed_index + 1
        while right < len(window_df) and low_mask[right]:
            selected[right] = True
            right += 1

    window_df["direction_jitter_seed"] = seed_mask.astype(bool)
    window_df["direction_jitter_selected"] = selected.astype(bool)
    window_df["direction_jitter_high_threshold"] = float(resolved_high_threshold)
    window_df["direction_jitter_low_threshold"] = float(resolved_low_threshold)
    return window_df


def add_direction_jitter_score_alpha(window_df: pd.DataFrame) -> pd.DataFrame:
    window_df = window_df.copy()
    if window_df.empty or "direction_jitter_score" not in window_df.columns:
        window_df["direction_jitter_seed"] = False
        window_df["direction_jitter_selected"] = False
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df

    window_df = mark_direction_jitter_hysteresis_windows(window_df)
    selected_scores = window_df.loc[window_df["direction_jitter_selected"], "direction_jitter_score"].to_numpy(dtype=float)
    finite = selected_scores[np.isfinite(selected_scores)]
    if finite.size == 0 or float(np.nanmax(finite)) <= float(FLOW_SCORE_MIN_VISIBLE):
        window_df["direction_jitter_score_alpha"] = 0.0
        return window_df

    lower = float(np.nanmin(finite))
    upper = float(np.nanmax(finite))
    scores = window_df["direction_jitter_score"].to_numpy(dtype=float)
    if upper <= lower:
        normalized = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), 1.0, 0.0)
    else:
        normalized = np.clip((np.nan_to_num(scores, nan=lower) - lower) / max(upper - lower, 1e-6), 0.0, 1.0)
    alpha = FLOW_SCORE_ALPHA_MIN + normalized * (FLOW_SCORE_ALPHA_MAX - FLOW_SCORE_ALPHA_MIN)
    alpha = np.where(window_df["direction_jitter_selected"].to_numpy(dtype=bool), alpha, 0.0)
    window_df["direction_jitter_score_alpha"] = alpha.astype(float)
    return window_df


def build_flow_direction_jitter_window_table(
    raw_flow_df: pd.DataFrame,
    camera: str,
    gy: int,
    gx: int,
    window_sec: float = FLOW_SCORE_WINDOW_SEC,
    hop_sec: float = FLOW_SCORE_HOP_SEC,
) -> pd.DataFrame:
    x_col = f"{camera}_flow_cell_{gy}_{gx}_x_mean"
    y_col = f"{camera}_flow_cell_{gy}_{gx}_y_mean"
    if raw_flow_df.empty or x_col not in raw_flow_df.columns or y_col not in raw_flow_df.columns or "time" not in raw_flow_df.columns:
        return pd.DataFrame()

    ordered = raw_flow_df.sort_values("time").reset_index(drop=True)
    direction_change = compute_flow_direction_change_abs(ordered, x_col, y_col).to_numpy(dtype=float)
    times = ordered["time"].astype(float).to_numpy()
    if times.size == 0:
        return pd.DataFrame()
    duration_sec = float(np.nanmax(times)) if np.isfinite(times).any() else 0.0
    duration_sec = max(duration_sec, float(window_sec))

    rows = []
    for start_sec in build_flow_window_starts(duration_sec=duration_sec, window_sec=window_sec, hop_sec=hop_sec):
        end_sec = float(min(start_sec + window_sec, duration_sec))
        start_sec = max(end_sec - window_sec, 0.0)
        mask = (times >= start_sec) & (times < end_sec)
        if not np.any(mask):
            nearest_index = int(np.argmin(np.abs(times - start_sec)))
            mask = np.zeros_like(times, dtype=bool)
            mask[nearest_index] = True
        segment = direction_change[mask]
        direction_mean = float(np.mean(segment))
        direction_sum = float(np.sum(segment))
        direction_p95 = float(np.percentile(segment, 95))
        direction_max = float(np.max(segment))
        direction_std = float(np.std(segment))
        direction_variation = float(np.mean(np.abs(np.diff(segment)))) if segment.size >= 2 else 0.0
        if direction_max > FLOW_SCORE_MIN_VISIBLE:
            high_threshold = direction_max * float(FLOW_SCORE_HIGH_RATIO_FRACTION)
            high_ratio = float(np.mean(segment >= high_threshold))
        else:
            high_ratio = 0.0
        rows.append({
            "camera": camera,
            "grid_x": int(gx + 1),
            "grid_y": int(gy + 1),
            "grid_col": int(gx),
            "grid_row": int(gy),
            "window_start_sec": float(start_sec),
            "window_end_sec": float(end_sec),
            "window_center_sec": float(0.5 * (start_sec + end_sec)),
            "direction_mean": direction_mean,
            "direction_sum": direction_sum,
            "direction_p95": direction_p95,
            "direction_max": direction_max,
            "direction_std": direction_std,
            "direction_variation": direction_variation,
            "direction_high_ratio": high_ratio,
        })

    window_df = pd.DataFrame(rows)
    for feature_name in ["direction_sum", "direction_high_ratio", "direction_variation", "direction_p95"]:
        window_df[f"{feature_name}_z"] = robust_positive_zscore(window_df[feature_name].to_numpy(dtype=float))
    window_df["direction_jitter_score_raw"] = (
        0.35 * window_df["direction_sum_z"]
        + 0.25 * window_df["direction_high_ratio_z"]
        + 0.25 * window_df["direction_variation_z"]
        + 0.15 * window_df["direction_p95_z"]
    ).astype(float)
    window_df["direction_jitter_score"] = percentile_normalize_0_1(
        window_df["direction_jitter_score_raw"].to_numpy(dtype=float)
    )
    return add_direction_jitter_score_alpha(window_df)


def build_flow_direction_jitter_score_table(raw_flow_df: pd.DataFrame, camera: str) -> pd.DataFrame:
    rows = []
    for gy in range(FLOW_GRID[0]):
        for gx in range(FLOW_GRID[1]):
            window_df = build_flow_direction_jitter_window_table(raw_flow_df, camera, gy, gx)
            if len(window_df):
                rows.append(window_df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def add_direction_jitter_topk_columns(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    required_cols = {"video_id", "camera", "window_start_sec", "direction_jitter_score"}
    missing_cols = sorted(required_cols - set(direction_jitter_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for direction jitter top-k: {missing_cols}")

    work = direction_jitter_df.copy()
    selected = work.get("direction_jitter_selected", pd.Series(False, index=work.index)).astype(bool)
    scores = pd.to_numeric(work["direction_jitter_score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    work["thresholded_direction_jitter_score"] = np.where(selected, scores, 0.0)
    work["direction_jitter_topk_rank"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
    work["direction_jitter_topk_selected"] = False
    work["window_start_bin"] = np.round(work["window_start_sec"].astype(float) / max(float(FLOW_SCORE_HOP_SEC), 1e-6)).astype(int)

    group_cols = ["video_id", "camera", "window_start_bin"]
    for _, group in work.groupby(group_cols, sort=False):
        ranked = group[group["thresholded_direction_jitter_score"].astype(float) > FLOW_SCORE_MIN_VISIBLE]
        ranked = ranked.sort_values("thresholded_direction_jitter_score", ascending=False).head(max(1, int(top_k)))
        if ranked.empty:
            continue
        work.loc[ranked.index, "direction_jitter_topk_rank"] = np.arange(1, len(ranked) + 1, dtype=int)
        work.loc[ranked.index, "direction_jitter_topk_selected"] = True
    return work


def build_broad_vibration_score_table(direction_jitter_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    """Average the top-k thresholded direction-jitter scores per camera for each time window."""
    if direction_jitter_df is None or direction_jitter_df.empty:
        return pd.DataFrame()
    required_cols = {"video_id", "camera", "window_start_sec", "window_end_sec", "window_center_sec", "direction_jitter_score"}
    missing_cols = sorted(required_cols - set(direction_jitter_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for broad vibration score: {missing_cols}")

    work = add_direction_jitter_topk_columns(direction_jitter_df, top_k=top_k)
    rows = []
    group_cols = ["video_id", "camera", "window_start_bin"]
    for keys, group in work.groupby(group_cols, sort=True):
        video_id, camera, window_start_bin = keys
        thresholded_scores = group["thresholded_direction_jitter_score"].to_numpy(dtype=float)
        top_values = np.sort(np.nan_to_num(thresholded_scores, nan=0.0, posinf=0.0, neginf=0.0))[::-1]
        top_k_values = np.zeros(max(1, int(top_k)), dtype=float)
        n_values = min(top_k_values.size, top_values.size)
        if n_values:
            top_k_values[:n_values] = top_values[:n_values]
        rows.append({
            "video_id": video_id,
            "camera": camera,
            "window_start_bin": int(window_start_bin),
            "window_start_sec": float(group["window_start_sec"].min()),
            "window_end_sec": float(group["window_end_sec"].max()),
            "window_center_sec": float(group["window_center_sec"].median()),
            "broad_vibration_score": float(top_k_values.mean()),
            "broad_vibration_top_k": int(top_k_values.size),
            "selected_grid_count": int(group.get("direction_jitter_selected", pd.Series(False, index=group.index)).astype(bool).sum()),
            "nonzero_top_k_count": int(np.count_nonzero(top_k_values > FLOW_SCORE_MIN_VISIBLE)),
            "max_thresholded_direction_jitter_score": float(np.max(top_values)) if top_values.size else 0.0,
        })
    return pd.DataFrame(rows).sort_values(["video_id", "camera", "window_start_sec"]).reset_index(drop=True)


def build_broad_vibration_feature_df(raw_flow_df: pd.DataFrame, top_k: int = BROAD_VIBRATION_TOP_K) -> pd.DataFrame:
    if raw_flow_df is None or raw_flow_df.empty:
        return pd.DataFrame(columns=["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS])

    raw_flow_df = raw_flow_df.copy()
    if "video_id" not in raw_flow_df.columns:
        raw_flow_df["video_id"] = "unknown"
    direction_tables = []
    for video_id, video_df in raw_flow_df.groupby("video_id", sort=False):
        for camera in ["front", "rear"]:
            direction_df = build_flow_direction_jitter_score_table(video_df, camera)
            if direction_df.empty:
                continue
            direction_df.insert(0, "video_id", video_id)
            direction_tables.append(direction_df)
    if not direction_tables:
        return pd.DataFrame(columns=["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS])

    direction_jitter_df = pd.concat(direction_tables, ignore_index=True)
    broad_df = build_broad_vibration_score_table(direction_jitter_df, top_k=top_k)
    if broad_df.empty:
        return pd.DataFrame(columns=["video_id", "time", "time_bin", *BROAD_VIBRATION_COLUMNS])

    broad_df = broad_df.copy()
    broad_df["time"] = broad_df["window_start_sec"].astype(float)
    broad_df["time_bin"] = np.round(broad_df["time"] / max(float(WINDOW_SEC), 1e-6)).astype(int)
    feature_df = broad_df.pivot_table(
        index=["video_id", "time_bin", "time"],
        columns="camera",
        values="broad_vibration_score",
        aggfunc="mean",
    ).reset_index()
    feature_df = feature_df.rename(columns={
        "front": "front_broad_vibration_score",
        "rear": "rear_broad_vibration_score",
    })
    feature_df.columns.name = None
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in feature_df.columns:
            feature_df[col] = 0.0
    return feature_df[["video_id", "time_bin", "time", *BROAD_VIBRATION_COLUMNS]].copy()


def ensure_broad_vibration_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in BROAD_VIBRATION_COLUMNS:
        if col not in df.columns:
            df[col] = 0.0
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
            if "video_id" in df.columns:
                df[col] = df.groupby("video_id")[col].transform(lambda s: s.ffill().bfill()).fillna(0.0)
            else:
                df[col] = df[col].fillna(0.0)
    return df


MODEL_EXCLUDE_COLS = {
    "time", "time_bin", "video_id", "sample_id", "label",
    "front_path", "rear_path", "audio_path",
}


def is_flow_xy_impact_score_column(column: str) -> bool:
    return column.endswith(FLOW_XY_SCORE_SUFFIXES)


def is_flow_xy_impact_diagnostic_column(column: str) -> bool:
    return (
        column.endswith(FLOW_XY_DIAGNOSTIC_SUFFIXES)
        or ("_flow_cell_" in column and column.endswith("_change_raw"))
    )


def infer_feature_group(column: str) -> str:
    if column in MODEL_EXCLUDE_COLS:
        return "metadata"
    if column.startswith("audio_mel_"):
        return "audio_mel"
    if column.startswith("audio_"):
        return "audio_basic"
    if column.startswith("front_rear_"):
        return "cross_camera"
    if column in BROAD_VIBRATION_COLUMNS or column.endswith("_broad_vibration_score"):
        return "broad_vibration"
    if is_flow_xy_impact_score_column(column):
        return "flow_change"
    if is_flow_xy_impact_diagnostic_column(column):
        return "flow_change_diagnostic"
    if "_flow_cell_" in column:
        return "flow_grid"
    if "_flow_" in column:
        return "flow_basic"
    if "_global_" in column:
        return "global_motion"
    if column in {"environment", "moving_state"}:
        return "context"
    if column == "sensor_missing" or column.startswith(("speed", "steering", "accel", "throttle", "brake")) or column.endswith("_flag"):
        return "sensor"
    return "other"


def build_feature_catalog(features_df: pd.DataFrame, feature_groups: dict[str, bool] = FEATURE_GROUPS) -> pd.DataFrame:
    rows = []
    for col in features_df.columns:
        group = infer_feature_group(col)
        rows.append({
            "column": col,
            "group": group,
            "enabled": bool(feature_groups.get(group, False)) and col not in FEATURE_EXCLUDE_COLUMNS,
            "dtype": str(features_df[col].dtype),
            "non_null": int(features_df[col].notna().sum()),
        })
    return pd.DataFrame(rows)


def select_model_source_columns(features_df: pd.DataFrame, feature_groups: dict[str, bool] = FEATURE_GROUPS) -> list[str]:
    selected = []
    for col in features_df.columns:
        group = infer_feature_group(col)
        if group == "metadata" or col in FEATURE_EXCLUDE_COLUMNS:
            continue
        if feature_groups.get(group, False):
            selected.append(col)
    return selected


def build_expanded_feature_group_map(source_columns: list[str], expanded_columns: list[str]) -> dict[str, str]:
    """Map one-hot expanded model columns back to their source feature group."""
    source_groups = {col: infer_feature_group(col) for col in source_columns}
    group_map: dict[str, str] = {}
    for expanded_col in expanded_columns:
        if expanded_col in source_groups:
            group_map[expanded_col] = source_groups[expanded_col]
            continue
        matched = None
        for source_col in source_columns:
            prefix = f"{source_col}_"
            if expanded_col.startswith(prefix):
                if matched is None or len(source_col) > len(matched):
                    matched = source_col
        group_map[expanded_col] = source_groups.get(matched, "other")
    return group_map


def build_feature_weight_vector(
    expanded_feature_names: np.ndarray,
    expanded_feature_group_map: dict[str, str],
    feature_group_weights: dict[str, float] = FEATURE_GROUP_WEIGHTS,
) -> np.ndarray:
    return np.asarray([
        float(feature_group_weights.get(expanded_feature_group_map.get(str(name), "other"), 1.0))
        for name in expanded_feature_names
    ], dtype=float)

## 10. 1サンプルの特徴量抽出

まず1本で特徴量の形と可視化を確認します。

In [11]:
def extract_sample_features(sample: pd.Series, sensor_csv_path: str | Path | None = None) -> pd.DataFrame:
    video_id = str(sample["sample_id"])
    feature_dfs = []
    raw_flow_dfs = []

    if sample.get("audio_path") is not pd.NA and Path(sample["audio_path"]).exists():
        feature_dfs.append(extract_audio_features(sample["audio_path"], video_id=video_id))

    if USE_FRONT and sample.get("front_path") is not pd.NA and Path(sample["front_path"]).exists():
        front_frames = extract_video_frames(sample["front_path"], fps_sample=FPS_SAMPLE, resize_width=FRAME_RESIZE_WIDTH)
        front_flow_df = compute_optical_flow_features(front_frames, "front", video_id=video_id)
        feature_dfs.append(front_flow_df)
        raw_flow_dfs.append(front_flow_df)
        feature_dfs.append(compute_global_motion_features(front_frames, "front", video_id=video_id))

    if USE_REAR and sample.get("rear_path") is not pd.NA and Path(sample["rear_path"]).exists():
        rear_frames = extract_video_frames(sample["rear_path"], fps_sample=FPS_SAMPLE, resize_width=FRAME_RESIZE_WIDTH)
        rear_flow_df = compute_optical_flow_features(rear_frames, "rear", video_id=video_id)
        feature_dfs.append(rear_flow_df)
        raw_flow_dfs.append(rear_flow_df)
        feature_dfs.append(compute_global_motion_features(rear_frames, "rear", video_id=video_id))

    if raw_flow_dfs:
        raw_flow_df = raw_flow_dfs[0]
        for other in raw_flow_dfs[1:]:
            raw_flow_df = raw_flow_df.merge(other.drop(columns=["time_bin"], errors="ignore"), on=["video_id", "time"], how="outer")
        raw_flow_df = raw_flow_df.sort_values(["video_id", "time"]).reset_index(drop=True)
        feature_dfs.append(build_broad_vibration_feature_df(raw_flow_df))

    feature_dfs.append(load_sensor_features(sensor_csv_path, video_id=video_id))
    df = align_features_by_time(feature_dfs)
    df = add_derived_features(df)
    df = ensure_broad_vibration_columns(df)
    df["environment"] = sample.get("environment", "unknown")
    df["label"] = "normal"
    return df


def _coerce_sample_id(value) -> str:
    return str(value).strip()


def select_balanced_train_samples(
    usable_samples: pd.DataFrame,
    max_train_videos: int | None = MAX_TRAIN_VIDEOS,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """Randomly select a near-equal number of usable videos from each environment."""
    if usable_samples.empty:
        return usable_samples.copy()

    if max_train_videos is None:
        return usable_samples.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    target_total = min(int(max_train_videos), len(usable_samples))
    env_counts = usable_samples["environment"].value_counts().sort_index()
    envs = env_counts.index.tolist()
    if not envs:
        return usable_samples.head(0).copy()

    base = target_total // len(envs)
    remainder = target_total % len(envs)
    quotas = {env: min(base, int(env_counts[env])) for env in envs}

    # 余りと、件数不足の環境から回収した枠を、まだ余力のある環境へ配る。
    assigned = sum(quotas.values())
    remaining = target_total - assigned
    rng = np.random.default_rng(random_state)
    env_order = envs.copy()
    rng.shuffle(env_order)
    while remaining > 0:
        progressed = False
        for env in env_order:
            if quotas[env] < int(env_counts[env]):
                quotas[env] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            break

    selected_parts = []
    for env in envs:
        n = quotas.get(env, 0)
        if n <= 0:
            continue
        env_df = usable_samples[usable_samples["environment"].eq(env)]
        stable_env_seed = sum((i + 1) * ord(ch) for i, ch in enumerate(str(env)))
        selected_parts.append(env_df.sample(n=n, random_state=random_state + stable_env_seed % 10000))

    if not selected_parts:
        return usable_samples.head(0).copy()
    return pd.concat(selected_parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)


def load_train_samples_from_list(sample_list_path: Path, usable_samples: pd.DataFrame) -> pd.DataFrame:
    saved = pd.read_csv(sample_list_path, dtype={"sample_id": str, "environment": str})
    if "sample_id" not in saved.columns:
        raise ValueError(f"sample list must contain sample_id column: {sample_list_path}")
    saved = saved.copy()
    saved["sample_id"] = saved["sample_id"].map(_coerce_sample_id)
    if "environment" in saved.columns:
        saved["environment"] = saved["environment"].astype(str)

    key_cols = ["sample_id", "environment"] if "environment" in saved.columns else ["sample_id"]
    restored = saved[key_cols].merge(usable_samples, on=key_cols, how="left", suffixes=("", "_current"))
    missing = restored["front_path"].isna() | restored["rear_path"].isna() | restored["audio_path"].isna()
    if missing.any():
        missing_rows = restored.loc[missing, key_cols].astype(str).agg("/".join, axis=1).tolist()
        raise FileNotFoundError(
            "Saved train sample list contains videos that are not usable now:\n"
            + "\n".join(missing_rows[:20])
        )
    return restored[usable_samples.columns].reset_index(drop=True)


def compute_balanced_environment_quotas(
    usable_samples: pd.DataFrame,
    target_total: int,
    random_state: int = RANDOM_STATE,
) -> dict[str, int]:
    env_counts = usable_samples["environment"].value_counts().sort_index()
    envs = env_counts.index.tolist()
    if not envs:
        return {}

    target_total = min(int(target_total), len(usable_samples))
    base = target_total // len(envs)
    quotas = {env: min(base, int(env_counts[env])) for env in envs}

    assigned = sum(quotas.values())
    remaining = target_total - assigned
    rng = np.random.default_rng(random_state)
    env_order = envs.copy()
    rng.shuffle(env_order)
    while remaining > 0:
        progressed = False
        for env in env_order:
            if quotas[env] < int(env_counts[env]):
                quotas[env] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            break
    return quotas


def reconcile_train_samples_to_max(
    saved_train_samples: pd.DataFrame,
    usable_samples: pd.DataFrame,
    max_train_videos: int | None = MAX_TRAIN_VIDEOS,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, str]:
    """Keep saved samples when possible, but resize the list to MAX_TRAIN_VIDEOS.

    When the saved list is too long, deterministic random rows are removed.
    When it is too short, deterministic random rows are added from usable samples not already selected.
    The adjustment also aims for the same indoor/outdoor ratio used for fresh selection.
    """
    saved_train_samples = saved_train_samples.drop_duplicates(["sample_id", "environment"]).reset_index(drop=True)
    if max_train_videos is None:
        return saved_train_samples, "saved_list"

    target_total = min(int(max_train_videos), len(usable_samples))
    if len(saved_train_samples) == target_total:
        return saved_train_samples, "saved_list"

    quotas = compute_balanced_environment_quotas(usable_samples, target_total, random_state=random_state)
    selected_parts = []
    selected_keys: set[tuple[str, str]] = set()

    for env, quota in quotas.items():
        saved_env = saved_train_samples[saved_train_samples["environment"].eq(env)]
        if len(saved_env) > quota:
            saved_env = saved_env.sample(n=quota, random_state=random_state).reset_index(drop=True)
        selected_parts.append(saved_env)
        selected_keys.update(zip(saved_env["sample_id"].astype(str), saved_env["environment"].astype(str)))

    selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else usable_samples.head(0).copy()

    for env, quota in quotas.items():
        current_n = int(selected["environment"].eq(env).sum()) if len(selected) else 0
        need = quota - current_n
        if need <= 0:
            continue
        pool = usable_samples[usable_samples["environment"].eq(env)].copy()
        pool = pool[~pool.apply(lambda r: (str(r["sample_id"]), str(r["environment"])) in selected_keys, axis=1)]
        if pool.empty:
            continue
        add_n = min(need, len(pool))
        stable_env_seed = sum((i + 1) * ord(ch) for i, ch in enumerate(str(env)))
        additions = pool.sample(n=add_n, random_state=random_state + stable_env_seed % 10000)
        selected = pd.concat([selected, additions], ignore_index=True)
        selected_keys.update(zip(additions["sample_id"].astype(str), additions["environment"].astype(str)))

    remaining = target_total - len(selected)
    if remaining > 0:
        pool = usable_samples[~usable_samples.apply(lambda r: (str(r["sample_id"]), str(r["environment"])) in selected_keys, axis=1)]
        if len(pool):
            additions = pool.sample(n=min(remaining, len(pool)), random_state=random_state)
            selected = pd.concat([selected, additions], ignore_index=True)

    if len(selected) > target_total:
        selected = selected.sample(n=target_total, random_state=random_state).reset_index(drop=True)
    else:
        selected = selected.sample(frac=1.0, random_state=random_state).reset_index(drop=True)

    action = "saved_list_resized"
    if len(saved_train_samples) < target_total:
        action = "saved_list_added"
    elif len(saved_train_samples) > target_total:
        action = "saved_list_trimmed"
    return selected[usable_samples.columns].reset_index(drop=True), action


def save_train_sample_list(train_samples: pd.DataFrame, sample_list_path: Path) -> None:
    sample_list_path.parent.mkdir(parents=True, exist_ok=True)
    cols = [
        "sample_id", "environment", "front_path", "rear_path", "audio_path",
        *[c for c in ["input_duration_sec", "status", "audio_status"] if c in train_samples.columns],
    ]
    out = train_samples[[c for c in cols if c in train_samples.columns]].copy()
    for col in ["front_path", "rear_path", "audio_path"]:
        if col in out.columns:
            out[col] = out[col].astype(str)
    out.to_csv(sample_list_path, index=False)


usable_samples = samples_df[samples_df["usable"]].copy()
usable_samples["sample_id"] = usable_samples["sample_id"].map(_coerce_sample_id)

if TRAIN_SAMPLE_LIST_PATH.exists():
    loaded_train_samples = load_train_samples_from_list(TRAIN_SAMPLE_LIST_PATH, usable_samples)
    train_samples, selection_source = reconcile_train_samples_to_max(loaded_train_samples, usable_samples, MAX_TRAIN_VIDEOS, RANDOM_STATE)
    if selection_source != "saved_list":
        save_train_sample_list(train_samples, TRAIN_SAMPLE_LIST_PATH)
else:
    train_samples = select_balanced_train_samples(usable_samples, MAX_TRAIN_VIDEOS, RANDOM_STATE)
    save_train_sample_list(train_samples, TRAIN_SAMPLE_LIST_PATH)
    selection_source = "balanced_random"

print(f"usable samples: {len(usable_samples)} / selected for this run: {len(train_samples)}")
print(f"selection source: {selection_source}")
print(f"train sample list: {TRAIN_SAMPLE_LIST_PATH}")
display(train_samples["environment"].value_counts().rename_axis("environment").reset_index(name="selected_count"))
display(train_samples[["sample_id", "environment", "front_path", "rear_path", "audio_path"]].head(20))

usable samples: 101 / selected for this run: 35
selection source: saved_list
train sample list: /workspaces/forklift/data/train_sample_list.csv


,environment,selected_count
0,outdoor,18
1,indoor,17


,sample_id,environment,front_path,rear_path,audio_path
0,1007,outdoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
1,1101,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
2,1008,outdoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
3,1001,outdoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
4,1047,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
5,1005,outdoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
6,1016,outdoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
7,1090,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
8,1075,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...
9,1035,indoor,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...,/workspaces/forklift/data/movie_preprocess/nor...


In [12]:
# 動作確認用に先頭1本だけ特徴量抽出します。
one_sample = train_samples.iloc[0]
one_features_df = extract_sample_features(one_sample)
print(one_features_df.shape)
display(one_features_df.head())
display(one_features_df.describe().T.head(30))

(78, 223)


,video_id,time_bin,audio_rms,audio_energy,audio_peak,audio_ptp,audio_zcr,audio_centroid,audio_bandwidth,audio_high_freq_energy,audio_missing,audio_mel_00,audio_mel_01,audio_mel_02,audio_mel_03,audio_mel_04,audio_mel_05,audio_mel_06,audio_mel_07,audio_mel_08,audio_mel_09,audio_mel_10,audio_mel_11,audio_mel_12,audio_mel_13,audio_mel_14,audio_mel_15,time,front_flow_mag_mean,front_flow_mag_std,front_flow_mag_max,front_flow_angle_mean,front_flow_angle_std,front_flow_x_mean,front_flow_x_std,front_flow_y_mean,front_flow_y_std,front_flow_failed,front_flow_cell_0_0_mag_mean,front_flow_cell_0_0_mag_std,front_flow_cell_0_0_x_mean,front_flow_cell_0_0_y_mean,front_flow_cell_0_0_valid_ratio,front_flow_cell_0_1_mag_mean,front_flow_cell_0_1_mag_std,front_flow_cell_0_1_x_mean,front_flow_cell_0_1_y_mean,front_flow_cell_0_1_valid_ratio,front_flow_cell_0_2_mag_mean,front_flow_cell_0_2_mag_std,front_flow_cell_0_2_x_mean,front_flow_cell_0_2_y_mean,front_flow_cell_0_2_valid_ratio,front_flow_cell_1_0_mag_mean,front_flow_cell_1_0_mag_std,front_flow_cell_1_0_x_mean,front_flow_cell_1_0_y_mean,front_flow_cell_1_0_valid_ratio,front_flow_cell_1_1_mag_mean,front_flow_cell_1_1_mag_std,...,rear_flow_cell_change_mean_raw,rear_flow_cell_change_max_raw,rear_flow_cell_change_p95_raw,rear_flow_valid_cell_count,rear_flow_cell_count,rear_flow_cell_0_0_change_raw,rear_flow_cell_0_1_change_raw,rear_flow_cell_0_2_change_raw,rear_flow_cell_1_0_change_raw,rear_flow_cell_1_1_change_raw,rear_flow_cell_1_2_change_raw,rear_flow_cell_2_0_change_raw,rear_flow_cell_2_1_change_raw,rear_flow_cell_2_2_change_raw,rear_global_dx,rear_global_dy,rear_global_translation_norm,rear_global_rotation,rear_global_scale,rear_global_ransac_residual,rear_global_inlier_ratio,rear_global_num_tracked_points,rear_global_failed,front_broad_vibration_score,rear_broad_vibration_score,sensor_missing,front_flow_change_amount_score,front_flow_globality_score,front_flow_transience_score,front_flow_xy_impact_score,front_flow_high_change_cell_count,front_flow_high_change_cell_ratio,front_flow_event_duration_steps,rear_flow_change_amount_score,rear_flow_globality_score,rear_flow_transience_score,rear_flow_xy_impact_score,rear_flow_high_change_cell_count,rear_flow_high_change_cell_ratio,rear_flow_event_duration_steps,front_rear_flow_mag_mean_mean,front_rear_flow_mag_mean_diff_abs,front_rear_flow_mag_mean_both_high,front_rear_flow_xy_impact_score_mean,front_rear_flow_xy_impact_score_diff_abs,front_rear_flow_xy_impact_score_both_high,front_rear_flow_globality_score_mean,front_rear_flow_globality_score_diff_abs,front_rear_flow_globality_score_both_high,front_rear_flow_change_amount_score_mean,front_rear_flow_change_amount_score_diff_abs,front_rear_flow_change_amount_score_both_high,front_rear_global_translation_norm_mean,front_rear_global_translation_norm_diff_abs,front_rear_global_translation_norm_both_high,front_rear_global_rotation_abs_mean,front_rear_global_rotation_abs_diff_abs,front_rear_global_rotation_abs_both_high,environment,label
0,1007,0,0.035488,0.001259,0.111638,0.219995,0.158487,1369.408319,183.206661,0.000380,0.0,-60.608757,-50.637871,-52.870033,-65.556671,-68.974731,-65.915482,-50.259247,-52.459560,-69.076431,-75.128464,-76.504303,-77.205185,-80.000000,-80.000000,-80.000000,-80.000000,0.0,1.523974,2.254281,12.725784,2.619699,1.814318,-0.098696,2.082517,0.248779,1.730829,0.0,2.727508,3.223405,-2.229727,1.320583,1.0,1.027695,1.612266,-0.028429,-0.819222,1.0,1.301969,0.620664,1.124807,0.578950,1.0,3.269147,3.499786,-1.915343,2.151235,1.0,1.381960,1.511708,...,0.000000,0.000000,0.000000,9.0,9.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.580502,-1.022553,1.175839,0.002517,1.000958,0.872175,0.737931,145.0,0.0,0.194494,0.630669,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.317932,0.412084,1.694492,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.865732,0.620214,0.653325,0.001602,0.001830,1.729943e-06,outdoor,normal
1,1007,1,0.059326,0.003520,0.228721,0.43880

,count,mean,std,min,25%,50%,75%,max
time_bin,78.0,38.500000,22.660538,0.000000e+00,19.250000,38.500000,57.750000,77.000000
audio_rms,78.0,0.067016,0.084297,1.091627e-13,0.011787,0.038136,0.079209,0.425010
audio_energy,78.0,0.011506,0.029162,1.191649e-26,0.000139,0.001457,0.006274,0.180633
audio_peak,78.0,0.255723,0.268244,1.323810e-12,0.060622,0.135698,0.375319,0.949279
audio_ptp,78.0,0.488849,0.507453,2.567808e-12,0.118659,0.263936,0.726133,1.809870
audio_zcr,78.0,0.128012,0.057012,2.313223e-02,0.093857,0.113160,0.160128,0.298531
audio_centroid,78.0,927.195841,580.182910,4.325864e-08,534.327291,785.790128,1106.424507,2874.111384
audio_bandwidth,78.0,545.258380,381.826894,1.506821e-02,283.116762,484.926453,651.588891,1833.509349
audio_high_freq_energy,78.0,0.027985,0.087710,6.622093e-12,0.000522,0.003605,0.011711,0.549251
audio_missing,78.0,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000


## 11. 特徴量の簡易可視化

In [13]:
def plot_feature_overview(features_df: pd.DataFrame, title: str | None = None):
    cols = [
        "audio_rms", "audio_high_freq_energy",
        "front_flow_mag_mean", "rear_flow_mag_mean",
        "front_flow_x_mean", "rear_flow_x_mean",
        "front_flow_y_mean", "rear_flow_y_mean",
        "front_flow_change_amount_score", "rear_flow_change_amount_score",
        "front_flow_globality_score", "rear_flow_globality_score",
        "front_flow_transience_score", "rear_flow_transience_score",
        "front_flow_xy_impact_score", "rear_flow_xy_impact_score",
        "front_flow_high_change_cell_ratio", "rear_flow_high_change_cell_ratio",
        "front_flow_event_duration_steps", "rear_flow_event_duration_steps",
        "front_global_translation_norm", "rear_global_translation_norm",
        "front_global_rotation", "rear_global_rotation",
    ]
    cols = [c for c in cols if c in features_df.columns]
    if not cols:
        print("No overview columns found.")
        return
    axes = features_df.plot(x="time", y=cols, subplots=True, figsize=(14, 2.2 * len(cols)), legend=True, title=title)
    plt.tight_layout()
    return axes

plot_feature_overview(one_features_df, title=f"sample {one_sample['sample_id']} feature overview")

array([<Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>,
       <Axes: xlabel='time'>, <Axes: xlabel='time'>], dtype=object)

## 12. 正常データ群から特徴量抽出

件数を増やすほど時間がかかります。最初は `MAX_TRAIN_VIDEOS=8` 程度で確認してください。

In [14]:
all_feature_dfs = []
for _, sample in tqdm(train_samples.iterrows(), total=len(train_samples), desc="extract features"):
    try:
        df = extract_sample_features(sample)
        all_feature_dfs.append(df)
    except Exception as exc:
        warnings.warn(f"feature extraction failed for sample_id={sample.get('sample_id')}: {exc}")

features_df = pd.concat(all_feature_dfs, ignore_index=True) if all_feature_dfs else pd.DataFrame()
flow_xy_impact_calibration = fit_flow_xy_impact_calibration(features_df)
features_df = apply_flow_xy_impact_calibration(features_df, flow_xy_impact_calibration)
features_df = add_cross_camera_features(features_df)
features_df = ensure_broad_vibration_columns(features_df)

print(features_df.shape)
print("broad vibration columns used by motion model:", [c for c in BROAD_VIBRATION_COLUMNS if c in features_df.columns])
display(pd.DataFrame([
    {
        "prefix": prefix,
        "amount_lower": stats.get("amount_scale", {}).get("lower"),
        "amount_upper": stats.get("amount_scale", {}).get("upper"),
        "cell_lower": stats.get("cell_scale", {}).get("lower"),
        "cell_upper": stats.get("cell_scale", {}).get("upper"),
        "cell_columns": len(stats.get("cell_columns", [])),
    }
    for prefix, stats in flow_xy_impact_calibration.get("prefix_stats", {}).items()
]))
display_cols = ["video_id", "time", *[c for c in BROAD_VIBRATION_COLUMNS if c in features_df.columns]]
display(features_df[display_cols].head())
if len(features_df):
    first_calibrated_video = str(features_df["video_id"].iloc[0])
    plot_feature_overview(
        features_df[features_df["video_id"].astype(str).eq(first_calibrated_video)],
        title=f"broad vibration / feature overview: video_id={first_calibrated_video}",
    )
display(features_df.head())
features_df.to_csv(OUTPUT_DIR / "features.csv", index=False)
print(f"saved: {OUTPUT_DIR / 'features.csv'}")

(2646, 223)
broad vibration columns used by motion model: ['front_broad_vibration_score', 'rear_broad_vibration_score']


,prefix,amount_lower,amount_upper,cell_lower,cell_upper,cell_columns
0,front,5.437179,33.327848,5.514419,54.801400,9
1,rear,5.140000,67.635893,5.667533,84.193563,9


,video_id,time,front_broad_vibration_score,rear_broad_vibration_score
0,1001,0.0,0.513730,0.570409
1,1001,0.2,0.513730,0.570409
2,1001,0.4,0.594613,0.998900
3,1001,0.6,0.594613,0.998900
4,1001,0.8,0.594613,0.998900


,video_id,time_bin,audio_rms,audio_energy,audio_peak,audio_ptp,audio_zcr,audio_centroid,audio_bandwidth,audio_high_freq_energy,audio_missing,audio_mel_00,audio_mel_01,audio_mel_02,audio_mel_03,audio_mel_04,audio_mel_05,audio_mel_06,audio_mel_07,audio_mel_08,audio_mel_09,audio_mel_10,audio_mel_11,audio_mel_12,audio_mel_13,audio_mel_14,audio_mel_15,time,front_flow_mag_mean,front_flow_mag_std,front_flow_mag_max,front_flow_angle_mean,front_flow_angle_std,front_flow_x_mean,front_flow_x_std,front_flow_y_mean,front_flow_y_std,front_flow_failed,front_flow_cell_0_0_mag_mean,front_flow_cell_0_0_mag_std,front_flow_cell_0_0_x_mean,front_flow_cell_0_0_y_mean,front_flow_cell_0_0_valid_ratio,front_flow_cell_0_1_mag_mean,front_flow_cell_0_1_mag_std,front_flow_cell_0_1_x_mean,front_flow_cell_0_1_y_mean,front_flow_cell_0_1_valid_ratio,front_flow_cell_0_2_mag_mean,front_flow_cell_0_2_mag_std,front_flow_cell_0_2_x_mean,front_flow_cell_0_2_y_mean,front_flow_cell_0_2_valid_ratio,front_flow_cell_1_0_mag_mean,front_flow_cell_1_0_mag_std,front_flow_cell_1_0_x_mean,front_flow_cell_1_0_y_mean,front_flow_cell_1_0_valid_ratio,front_flow_cell_1_1_mag_mean,front_flow_cell_1_1_mag_std,...,rear_flow_cell_change_mean_raw,rear_flow_cell_change_max_raw,rear_flow_cell_change_p95_raw,rear_flow_valid_cell_count,rear_flow_cell_count,rear_flow_cell_0_0_change_raw,rear_flow_cell_0_1_change_raw,rear_flow_cell_0_2_change_raw,rear_flow_cell_1_0_change_raw,rear_flow_cell_1_1_change_raw,rear_flow_cell_1_2_change_raw,rear_flow_cell_2_0_change_raw,rear_flow_cell_2_1_change_raw,rear_flow_cell_2_2_change_raw,rear_global_dx,rear_global_dy,rear_global_translation_norm,rear_global_rotation,rear_global_scale,rear_global_ransac_residual,rear_global_inlier_ratio,rear_global_num_tracked_points,rear_global_failed,front_broad_vibration_score,rear_broad_vibration_score,sensor_missing,front_flow_change_amount_score,front_flow_globality_score,front_flow_transience_score,front_flow_xy_impact_score,front_flow_high_change_cell_count,front_flow_high_change_cell_ratio,front_flow_event_duration_steps,rear_flow_change_amount_score,rear_flow_globality_score,rear_flow_transience_score,rear_flow_xy_impact_score,rear_flow_high_change_cell_count,rear_flow_high_change_cell_ratio,rear_flow_event_duration_steps,front_rear_flow_mag_mean_mean,front_rear_flow_mag_mean_diff_abs,front_rear_flow_mag_mean_both_high,front_rear_flow_xy_impact_score_mean,front_rear_flow_xy_impact_score_diff_abs,front_rear_flow_xy_impact_score_both_high,front_rear_flow_globality_score_mean,front_rear_flow_globality_score_diff_abs,front_rear_flow_globality_score_both_high,front_rear_flow_change_amount_score_mean,front_rear_flow_change_amount_score_diff_abs,front_rear_flow_change_amount_score_both_high,front_rear_global_translation_norm_mean,front_rear_global_translation_norm_diff_abs,front_rear_global_translation_norm_both_high,front_rear_global_rotation_abs_mean,front_rear_global_rotation_abs_diff_abs,front_rear_global_rotation_abs_both_high,environment,label
0,1001,0,0.065897,0.004342,0.375093,0.742492,0.063457,377.993243,198.174505,0.001628,0.0,-16.660177,-11.645163,-16.637348,-35.360619,-38.626770,-40.808693,-42.747826,-45.462425,-44.120560,-47.834698,-47.634502,-47.754623,-49.076714,-60.596264,-60.549896,-64.325912,0.0,1.384651,1.841498,15.849631,2.374206,1.734207,-0.224015,1.746961,0.793268,1.255805,0.0,0.452363,0.463062,-0.376288,0.180682,1.0,0.178940,0.324090,0.014864,-0.086168,1.0,0.592928,0.238786,0.547387,0.002466,1.0,2.245057,1.245583,-0.983803,1.586625,1.0,1.728886,3.269217,...,0.000000,0.000000,0.000000,9.0,9.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.001448,-0.000246,0.001468,-0.000001,1.000007,0.005584,1.0000,200.0,0.0,0.513730,0.570409,1.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.718494,1.332315,0.072467,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.433979,0.865021,0.001272,0.0003

saved: /workspaces/forklift/outputs/anomaly_feature_poc/features.csv


## 13. 前処理と Isolation Forest 学習

In [15]:
def preprocess_features(
    features_df: pd.DataFrame,
    feature_groups: dict[str, bool] = FEATURE_GROUPS,
    feature_group_weights: dict[str, float] = FEATURE_GROUP_WEIGHTS,
):
    feature_catalog = build_feature_catalog(features_df, feature_groups=feature_groups)
    selected_source_cols = select_model_source_columns(features_df, feature_groups=feature_groups)
    if not selected_source_cols:
        raise ValueError("No feature columns were selected for this model. Check SCORE_MODEL_CONFIGS.")

    model_df = features_df[selected_source_cols].copy()
    model_df = pd.get_dummies(model_df, columns=model_df.select_dtypes(include=["object", "category"]).columns, dummy_na=True)
    model_df = model_df.replace([np.inf, -np.inf], np.nan)

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X_scaled = pipeline.fit_transform(model_df)
    feature_names = model_df.columns.to_numpy()
    expanded_feature_group_map = build_expanded_feature_group_map(selected_source_cols, feature_names.tolist())
    feature_weight_vector = build_feature_weight_vector(feature_names, expanded_feature_group_map, feature_group_weights)
    X = X_scaled * feature_weight_vector
    return X, X_scaled, pipeline, feature_names, model_df, feature_catalog, selected_source_cols, expanded_feature_group_map, feature_weight_vector


def train_isolation_forest(X: np.ndarray, contamination=CONTAMINATION, random_state: int = RANDOM_STATE) -> IsolationForest:
    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        max_samples="auto",
        random_state=random_state,
        n_jobs=-1,
    )
    model.fit(X)
    return model


def compute_anomaly_scores(model: IsolationForest, X: np.ndarray) -> np.ndarray:
    return -model.decision_function(X)


def fit_score_calibration(scores: np.ndarray, quantiles: tuple[float, float] = SCORE_CALIBRATION_QUANTILES) -> dict[str, float]:
    finite = np.asarray(scores, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return {"lower": 0.0, "upper": 1.0}
    lower_q, upper_q = quantiles
    lower = float(np.quantile(finite, lower_q))
    upper = float(np.quantile(finite, upper_q))
    if not np.isfinite(upper) or upper <= lower:
        upper = lower + 1e-6
    return {"lower": lower, "upper": upper}


def apply_score_calibration(scores: np.ndarray | pd.Series, calibration: dict[str, float]) -> np.ndarray:
    values = np.asarray(scores, dtype=float)
    lower = float(calibration.get("lower", 0.0))
    upper = float(calibration.get("upper", lower + 1.0))
    denom = max(upper - lower, 1e-6)
    return np.clip((values - lower) / denom, 0.0, 1.0)


def compute_temporal_sync_score(
    scored_df: pd.DataFrame,
    audio_col: str = SYNC_SCORE_CONFIG["audio_score_column"],
    motion_col: str = SYNC_SCORE_CONFIG["motion_score_column"],
    max_lag_windows: int = SYNC_SCORE_CONFIG["max_lag_windows"],
) -> pd.Series:
    if audio_col not in scored_df.columns or motion_col not in scored_df.columns:
        return pd.Series(0.0, index=scored_df.index, name="sync_score")

    sync = pd.Series(0.0, index=scored_df.index, name="sync_score")
    sort_col = "time_bin" if "time_bin" in scored_df.columns else "time"
    window = max(1, int(max_lag_windows) * 2 + 1)

    for _, group in scored_df.sort_values(["video_id", sort_col]).groupby("video_id", sort=False):
        audio = group[audio_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        motion = group[motion_col].astype(float).fillna(0.0).clip(0.0, 1.0)
        nearby_audio = audio.rolling(window=window, center=True, min_periods=1).max()
        nearby_motion = motion.rolling(window=window, center=True, min_periods=1).max()
        aligned = np.maximum(audio * nearby_motion, motion * nearby_audio)
        sync.loc[group.index] = np.sqrt(np.clip(aligned, 0.0, 1.0))
    return sync


def add_composite_scores(
    scored_df: pd.DataFrame,
    final_score_weights: dict[str, float] = FINAL_SCORE_WEIGHTS,
    sync_config: dict = SYNC_SCORE_CONFIG,
) -> pd.DataFrame:
    scored_df = scored_df.copy()
    scored_df["sync_score"] = compute_temporal_sync_score(
        scored_df,
        audio_col=sync_config.get("audio_score_column", "audio_anomaly_score"),
        motion_col=sync_config.get("motion_score_column", "motion_anomaly_score"),
        max_lag_windows=int(sync_config.get("max_lag_windows", 2)),
    )

    available_weights = {col: float(weight) for col, weight in final_score_weights.items() if col in scored_df.columns and float(weight) > 0}
    weight_sum = sum(available_weights.values())
    if weight_sum <= 0:
        scored_df["final_anomaly_score"] = 0.0
    else:
        final_score = np.zeros(len(scored_df), dtype=float)
        for col, weight in available_weights.items():
            final_score += scored_df[col].astype(float).fillna(0.0).to_numpy() * (weight / weight_sum)
        scored_df["final_anomaly_score"] = np.clip(final_score, 0.0, 1.0)

    # 既存出力との互換用。以後の anomaly_score は合成後の最終スコアです。
    scored_df["anomaly_score"] = scored_df["final_anomaly_score"]
    scored_df["anomaly_score_smooth"] = scored_df.groupby("video_id")["anomaly_score"].transform(
        lambda s: s.rolling(window=5, center=True, min_periods=1).mean()
    )
    return scored_df


def train_score_model(model_name: str, config: dict, features_df: pd.DataFrame) -> tuple[dict, np.ndarray, np.ndarray, np.ndarray]:
    feature_groups = config.get("feature_groups", FEATURE_GROUPS)
    feature_group_weights = config.get("feature_group_weights", FEATURE_GROUP_WEIGHTS)
    (
        X,
        X_scaled_unweighted,
        preprocess_pipeline,
        feature_names,
        model_input_df,
        feature_catalog,
        selected_source_cols,
        expanded_feature_group_map,
        feature_weight_vector,
    ) = preprocess_features(features_df, feature_groups=feature_groups, feature_group_weights=feature_group_weights)
    model = train_isolation_forest(X)
    raw_scores = compute_anomaly_scores(model, X)
    score_calibration = fit_score_calibration(raw_scores)
    calibrated_scores = apply_score_calibration(raw_scores, score_calibration)
    artifact = {
        "model_name": model_name,
        "model": model,
        "preprocess_pipeline": preprocess_pipeline,
        "feature_names": feature_names,
        "feature_groups": feature_groups,
        "feature_group_weights": feature_group_weights,
        "feature_exclude_columns": FEATURE_EXCLUDE_COLUMNS,
        "selected_source_columns": selected_source_cols,
        "expanded_feature_group_map": expanded_feature_group_map,
        "feature_weight_vector": feature_weight_vector,
        "feature_catalog": feature_catalog,
        "score_calibration": score_calibration,
        "score_column": config.get("score_column", f"{model_name}_anomaly_score"),
        "raw_score_column": config.get("raw_score_column", f"{model_name}_anomaly_score_raw"),
        "model_input_shape": X.shape,
    }
    return artifact, X, raw_scores, calibrated_scores


score_model_artifacts: dict[str, dict] = {}
model_inputs_by_name: dict[str, np.ndarray] = {}
model_feature_catalogs = []

for model_name, config in SCORE_MODEL_CONFIGS.items():
    if not config.get("enabled", True):
        continue
    artifact, X_model, raw_scores, calibrated_scores = train_score_model(model_name, config, features_df)
    score_model_artifacts[model_name] = artifact
    model_inputs_by_name[model_name] = X_model
    features_df[artifact["raw_score_column"]] = raw_scores
    features_df[artifact["score_column"]] = calibrated_scores

    catalog = artifact["feature_catalog"].copy()
    catalog.insert(0, "model_name", model_name)
    model_feature_catalogs.append(catalog)

features_df = add_composite_scores(features_df)
feature_catalog_df = pd.concat(model_feature_catalogs, ignore_index=True) if model_feature_catalogs else pd.DataFrame()
primary_model_name = next(iter(score_model_artifacts))
primary_artifact = score_model_artifacts[primary_model_name]
X = model_inputs_by_name[primary_model_name]
feature_names = np.asarray(primary_artifact["feature_names"])

print("score models:", list(score_model_artifacts.keys()))
for name, artifact in score_model_artifacts.items():
    print(f"- {name}: input_shape={artifact['model_input_shape']}, score_calibration={artifact['score_calibration']}")
print("sync score config:", SYNC_SCORE_CONFIG)
print("final score weights:", FINAL_SCORE_WEIGHTS)
summary_cols = ["model_name", "group", "enabled"] if len(feature_catalog_df) else []
if summary_cols:
    display(feature_catalog_df.groupby(summary_cols).size().reset_index(name="column_count"))
    expanded_summaries = []
    for name, artifact in score_model_artifacts.items():
        expanded_summaries.append(pd.DataFrame({
            "model_name": name,
            "feature": artifact["feature_names"],
            "group": [artifact["expanded_feature_group_map"].get(str(c), "other") for c in artifact["feature_names"]],
            "weight": artifact["feature_weight_vector"],
        }))
    display(pd.concat(expanded_summaries, ignore_index=True).groupby(["model_name", "group", "weight"]).size().reset_index(name="expanded_column_count"))
display(features_df[["video_id", "time", "audio_anomaly_score", "motion_anomaly_score", "sync_score", "anomaly_score", "anomaly_score_smooth"]].head())

score models: ['audio', 'motion']
- audio: input_shape=(2646, 25), score_calibration={'lower': -0.12479200401559061, 'upper': 0.11095725310548356}
- motion: input_shape=(2646, 2), score_calibration={'lower': -0.1120229803879298, 'upper': 0.1743621024353136}
sync score config: {'audio_score_column': 'audio_anomaly_score', 'motion_score_column': 'motion_anomaly_score', 'max_lag_windows': 2}
final score weights: {'audio_anomaly_score': 0.45, 'motion_anomaly_score': 0.35, 'sync_score': 0.2}


,model_name,group,enabled,column_count
0,audio,audio_basic,True,9
1,audio,audio_mel,True,16
2,audio,broad_vibration,False,2
3,audio,context,False,1
4,audio,cross_camera,False,18
5,audio,flow_basic,False,20
6,audio,flow_change,False,8
7,audio,flow_change_diagnostic,False,36
8,audio,flow_grid,False,90
9,audio,global_motion,False,18


,model_name,group,weight,expanded_column_count
0,audio,audio_basic,1.0,9
1,audio,audio_mel,1.0,16
2,motion,broad_vibration,1.0,2


,video_id,time,audio_anomaly_score,motion_anomaly_score,sync_score,anomaly_score,anomaly_score_smooth
0,1001,0.0,0.290301,0.541911,0.688066,0.457918,0.663801
1,1001,0.2,0.614252,0.541911,0.769065,0.619895,0.722596
2,1001,0.4,0.873640,0.962898,0.917184,0.913589,0.723833
3,1001,0.6,0.823403,0.962898,0.957184,0.898982,0.790153
4,1001,0.8,0.445179,0.962898,0.957184,0.728782,0.810515


## 14. 動画単位スコアと上位異常候補

In [16]:
def summarize_video_scores(scored_df: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    def topk_mean(s):
        return s.nlargest(min(top_k, len(s))).mean()

    agg_spec = {
        "max_anomaly_score": ("anomaly_score", "max"),
        "topk_mean_anomaly_score": ("anomaly_score", topk_mean),
        "p95_anomaly_score": ("anomaly_score", lambda s: s.quantile(0.95)),
        "n_windows": ("anomaly_score", "size"),
    }
    for col in ["audio_anomaly_score", "motion_anomaly_score", "sync_score"]:
        if col in scored_df.columns:
            agg_spec[f"max_{col}"] = (col, "max")
            agg_spec[f"topk_mean_{col}"] = (col, topk_mean)
    return scored_df.groupby("video_id").agg(**agg_spec).sort_values("max_anomaly_score", ascending=False).reset_index()

video_scores_df = summarize_video_scores(features_df)
display(video_scores_df)

score_cols = [c for c in PLOT_SCORE_COLUMNS if c in features_df.columns]
feature_cols = [c for c in PLOT_FEATURE_COLUMNS if c in features_df.columns]
top_anomalies_df = features_df.nlargest(TOP_N_ANOMALIES, "anomaly_score")[[
    "video_id", "time", *score_cols, "anomaly_score_smooth", *feature_cols,
]]
display(top_anomalies_df)

,video_id,max_anomaly_score,topk_mean_anomaly_score,p95_anomaly_score,n_windows,max_audio_anomaly_score,topk_mean_audio_anomaly_score,max_motion_anomaly_score,topk_mean_motion_anomaly_score,max_sync_score,topk_mean_sync_score
0,1007,0.998284,0.860018,0.716371,78,1.000000,0.902287,1.000000,0.997058,1.000000,0.999018
1,1002,0.969223,0.855292,0.736892,77,1.000000,0.962652,1.000000,0.992415,0.970249,0.930321
2,1004,0.940541,0.853418,0.792590,73,0.992997,0.882744,0.880835,0.878669,0.933318,0.918947
3,1001,0.913589,0.830872,0.756113,72,1.000000,1.000000,0.962898,0.952696,0.957184,0.936449
4,1070,0.895884,0.763106,0.673091,77,0.954346,0.852698,1.000000,1.000000,0.901369,0.901369
5,1011,0.889371,0.821783,0.746119,78,0.920298,0.851782,0.958870,0.923558,0.907488,0.882042
6,1012,0.887078,0.802762,0.717695,72,1.000000,0.885036,0.752950,0.748253,0.867727,0.850850
7,1017,0.880912,0.809438,0.780689,74,1.000000,0.839751,0.988470,0.946368,0.892322,0.888905
8,1075,0.875234,0.763411,0.685465,78,1.000000,0.946466,1.000000,1.000000,0.880760,0.866871
9,1034,0.871771,0.690453,0.561412,75,0.908102,0.714658,0.827780,0.793609,0.867011,0.811418


,video_id,time,anomaly_score,audio_anomaly_score,motion_anomaly_score,sync_score,anomaly_score_smooth,audio_rms,audio_high_freq_energy,front_broad_vibration_score,rear_broad_vibration_score,front_global_translation_norm,rear_global_translation_norm,front_flow_mag_mean,rear_flow_mag_mean,front_flow_x_mean,rear_flow_x_mean,front_flow_y_mean,rear_flow_y_mean,front_flow_change_amount_score,rear_flow_change_amount_score,front_flow_globality_score,rear_flow_globality_score,front_flow_transience_score,rear_flow_transience_score,front_flow_xy_impact_score,rear_flow_xy_impact_score,front_flow_high_change_cell_ratio,rear_flow_high_change_cell_ratio,front_flow_event_duration_steps,rear_flow_event_duration_steps
498,1007,10.4,0.998284,1.000000,0.995097,1.000000,0.860018,0.425010,0.158417,1.000000,0.827128,0.986563,0.903729,3.676510,2.905308,0.205952,1.422054,1.962371,-1.719932,0.427627,0.146299,0.231232,0.223984,0.0,0.0,0.201141,0.141252,0.222222,0.222222,0.0,0.0
81,1002,1.8,0.969223,0.959579,0.981038,0.970249,0.763310,0.403149,0.062830,0.993889,0.269877,0.526105,29.624136,2.586680,6.629615,1.147414,4.871834,0.162683,-1.814128,0.231081,0.217119,0.223892,0.117557,0.0,0.0,0.158162,0.102202,0.222222,0.111111,0.0,0.0
80,1002,1.6,0.957695,0.933960,0.981038,0.970249,0.809971,0.437044,0.036086,0.993889,0.269877,0.872638,29.144320,2.699764,7.837086,1.640750,4.357578,0.425974,-2.768553,0.194408,0.654274,0.229458,0.231144,0.0,1.0,0.153611,0.546427,0.222222,0.222222,0.0,1.0
268,1004,8.8,0.940541,0.992997,0.877225,0.933318,0.824014,0.302470,0.196743,0.949090,0.364239,0.576615,1.750182,1.260704,2.449835,0.351381,1.764258,-0.225591,0.996126,0.033419,0.000000,0.115724,0.000000,0.0,0.0,0.064546,0.000000,0.111111,0.000000,0.0,0.0
2,1001,0.4,0.913589,0.873640,0.962898,0.917184,0.723833,0.359258,0.022459,0.594613,0.998900,0.757358,0.285400,2.309355,0.635710,0.430511,-0.011100,-0.968403,-0.400156,0.985158,0.052792,0.336214,0.000000,1.0,0.0,0.665138,0.010558,0.333333,0.000000,1.0,0.0
499,1007,10.6,0.906987,0.798210,0.995097,0.997546,0.755667,0.222063,0.028353,1.000000,0.827128,1.175387,3.196141,2.567353,1.412048,1.111600,-0.018635,-1.400204,0.452603,0.484013,0.210895,0.338589,0.119535,0.0,0.0,0.266097,0.101946,0.333333,0.111111,0.0,0.0
3,1001,0.6,0.898982,0.823403,0.962898,0.957184,0.790153,0.312415,0.003622,0.594613,0.998900,0.531679,0.478034,2.941669,0.702771,1.342035,-0.019036,-1.982872,0.467454,0.201890,0.071716,0.220156,0.112646,0.0,0.0,0.150456,0.070666,0.222222,0.111111,0.0,0.0
2111,1070,14.4,0.895884,0.812466,1.000000,0.901369,0.763106,0.010258,0.606481,1.000000,1.000000,0.109142,0.158685,0.144936,0.084709,-0.030334,-0.017868,0.040603,0.004791,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0


## 15. 異常スコアの可視化

In [17]:
def plot_anomaly_scores(scored_df: pd.DataFrame, video_id: str | None = None, save_path: str | Path | None = None):
    if video_id is None:
        video_id = str(scored_df["video_id"].iloc[0])
    df = scored_df[scored_df["video_id"].astype(str).eq(str(video_id))].copy()
    if df.empty:
        print(f"video_id not found: {video_id}")
        return None

    cols = [c for c in [*PLOT_SCORE_COLUMNS, *PLOT_FEATURE_COLUMNS] if c in df.columns]
    axes = df.plot(x="time", y=cols, subplots=True, figsize=(14, 2.2 * len(cols)), title=f"video_id={video_id}")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"saved: {save_path}")
    return axes

first_scored_video = str(video_scores_df.iloc[0]["video_id"]) if len(video_scores_df) else None
if first_scored_video is not None:
    plot_anomaly_scores(features_df, first_scored_video, OUTPUT_DIR / "anomaly_plot.png")

saved: /workspaces/forklift/outputs/anomaly_feature_poc/anomaly_plot.png


## 16. 簡易的な特徴量寄与確認

Isolation Forest の厳密な寄与ではありません。標準化後の絶対値が大きい特徴を見ることで、ピーク時刻で何が通常分布から外れていたかをデバッグします。

In [18]:
def get_top_contributing_features(model_name: str, row_index: int, top_n: int = 10) -> pd.DataFrame:
    artifact = score_model_artifacts[model_name]
    X_model = model_inputs_by_name[model_name]
    values = X_model[row_index]
    feature_names = np.asarray(artifact["feature_names"])
    order = np.argsort(np.abs(values))[::-1][:top_n]
    return pd.DataFrame({
        "model_name": model_name,
        "feature": feature_names[order],
        "group": [artifact["expanded_feature_group_map"].get(str(feature_names[i]), "other") for i in order],
        "scaled_value": values[order],
        "abs_scaled_value": np.abs(values[order]),
    })

peak_idx = int(features_df["anomaly_score"].idxmax())
peak_pos = features_df.index.get_loc(peak_idx)
print(features_df.loc[peak_idx, ["video_id", "time", "audio_anomaly_score", "motion_anomaly_score", "sync_score", "anomaly_score", "anomaly_score_smooth"]])
for model_name in score_model_artifacts:
    display(get_top_contributing_features(model_name, peak_pos, top_n=12))

video_id                    1007
time                        10.4
audio_anomaly_score          1.0
motion_anomaly_score    0.995097
sync_score                   1.0
anomaly_score           0.998284
anomaly_score_smooth    0.860018
Name: 498, dtype: object


,model_name,feature,group,scaled_value,abs_scaled_value
0,audio,audio_energy,audio_basic,8.341138,8.341138
1,audio,audio_rms,audio_basic,5.462518,5.462518
2,audio,audio_mel_15,audio_mel,3.461686,3.461686
3,audio,audio_ptp,audio_basic,3.419460,3.419460
4,audio,audio_peak,audio_basic,3.285338,3.285338
5,audio,audio_mel_14,audio_mel,3.240539,3.240539
6,audio,audio_mel_13,audio_mel,3.045739,3.045739
7,audio,audio_mel_12,audio_mel,2.953400,2.953400
8,audio,audio_bandwidth,audio_basic,2.773101,2.773101
9,audio,audio_mel_11,audio_mel,2.389483,2.389483


,model_name,feature,group,scaled_value,abs_scaled_value
0,motion,front_broad_vibration_score,broad_vibration,2.991436,2.991436
1,motion,rear_broad_vibration_score,broad_vibration,2.288772,2.288772


## 17. 異常ピーク時刻のフレーム確認

In [19]:
def read_frame_at_time(video_path: str | Path, time_sec: float) -> np.ndarray | None:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, time_sec) * 1000)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def show_frames_around_time(video_path: str | Path, time_sec: float, offsets=(-0.4, 0.0, 0.4), title: str | None = None):
    frames = []
    labels = []
    for offset in offsets:
        t = max(0.0, time_sec + offset)
        frame = read_frame_at_time(video_path, t)
        if frame is not None:
            frames.append(frame)
            labels.append(f"t={t:.2f}s")
    if not frames:
        print(f"No frames loaded from {video_path}")
        return
    fig, axes = plt.subplots(1, len(frames), figsize=(5 * len(frames), 4))
    if len(frames) == 1:
        axes = [axes]
    for ax, frame, label in zip(axes, frames, labels):
        ax.imshow(frame)
        ax.set_title(label)
        ax.axis("off")
    if title:
        fig.suptitle(title)
    plt.tight_layout()

peak_video_id = str(features_df.loc[peak_idx, "video_id"])
peak_time = float(features_df.loc[peak_idx, "time"])
peak_sample = samples_df[samples_df["sample_id"].astype(str).eq(peak_video_id)].iloc[0]
print(f"peak video_id={peak_video_id}, time={peak_time:.2f}s")
if USE_FRONT:
    show_frames_around_time(peak_sample["front_path"], peak_time, title="front")
if USE_REAR:
    show_frames_around_time(peak_sample["rear_path"], peak_time, title="rear")

peak video_id=1007, time=10.40s


## 18. 結果保存

In [20]:
def save_outputs(scored_df: pd.DataFrame, output_dir: str | Path = OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    scored_df.to_csv(output_dir / "anomaly_scores.csv", index=False)
    video_scores_df.to_csv(output_dir / "video_scores.csv", index=False)
    feature_catalog_df.to_csv(output_dir / "feature_catalog.csv", index=False)
    for model_name, artifact in score_model_artifacts.items():
        artifact["feature_catalog"].to_csv(output_dir / f"feature_catalog_{model_name}.csv", index=False)

    primary_artifact = score_model_artifacts[next(iter(score_model_artifacts))]
    joblib.dump({
        "model_version": "audio_motion_sync_v9_broad_vibration",
        "score_models": score_model_artifacts,
        "score_model_configs": SCORE_MODEL_CONFIGS,
        "sync_score_config": SYNC_SCORE_CONFIG,
        "final_score_weights": FINAL_SCORE_WEIGHTS,
        "plot_score_columns": PLOT_SCORE_COLUMNS,
        "plot_feature_columns": PLOT_FEATURE_COLUMNS,
        "feature_groups": FEATURE_GROUPS,
        "feature_group_weights": FEATURE_GROUP_WEIGHTS,
        "feature_exclude_columns": FEATURE_EXCLUDE_COLUMNS,
        "feature_catalog": feature_catalog_df,
        "flow_xy_impact_config": FLOW_XY_IMPACT_CONFIG,
        "flow_xy_impact_columns": FLOW_XY_IMPACT_COLUMNS,
        "flow_xy_impact_calibration": flow_xy_impact_calibration,
        "broad_vibration_columns": BROAD_VIBRATION_COLUMNS,
        "broad_vibration_top_k": BROAD_VIBRATION_TOP_K,
        "settings": {
            "fps_sample": FPS_SAMPLE,
            "window_sec": WINDOW_SEC,
            "audio_sr": AUDIO_SR,
            "n_mels": N_MELS,
            "frame_resize_width": FRAME_RESIZE_WIDTH,
            "flow_analysis_scale": FLOW_ANALYSIS_SCALE,
            "flow_grid": FLOW_GRID,
            "motion_feature": "broad_vibration",
            "flow_change_feature": "xy_impact",
            "flow_min_valid_cell_ratio": FLOW_MIN_VALID_CELL_RATIO,
            "flow_score_lower_quantile": FLOW_SCORE_LOWER_QUANTILE,
            "flow_score_upper_quantile": FLOW_SCORE_UPPER_QUANTILE,
            "flow_cell_score_upper_quantile": FLOW_CELL_SCORE_UPPER_QUANTILE,
            "flow_event_threshold": FLOW_EVENT_THRESHOLD,
            "flow_duration_alpha": FLOW_DURATION_ALPHA,
            "flow_globality_weight": FLOW_GLOBALITY_WEIGHT,
            "flow_transience_weight": FLOW_TRANSIENCE_WEIGHT,
            "flow_change_amount_weight": FLOW_CHANGE_AMOUNT_WEIGHT,
            "flow_score_window_sec": FLOW_SCORE_WINDOW_SEC,
            "flow_score_hop_sec": FLOW_SCORE_HOP_SEC,
            "flow_score_alpha_min": FLOW_SCORE_ALPHA_MIN,
            "flow_score_alpha_max": FLOW_SCORE_ALPHA_MAX,
            "flow_score_min_visible": FLOW_SCORE_MIN_VISIBLE,
            "flow_score_high_ratio_fraction": FLOW_SCORE_HIGH_RATIO_FRACTION,
            "flow_direction_min_mag": FLOW_DIRECTION_MIN_MAG,
            "direction_jitter_high_threshold": DIRECTION_JITTER_HIGH_THRESHOLD,
            "direction_jitter_low_threshold": DIRECTION_JITTER_LOW_THRESHOLD,
            "direction_jitter_threshold_mode": DIRECTION_JITTER_THRESHOLD_MODE,
            "direction_jitter_high_percentile": DIRECTION_JITTER_HIGH_PERCENTILE,
            "direction_jitter_low_percentile": DIRECTION_JITTER_LOW_PERCENTILE,
            "direction_jitter_low_expansion_steps": DIRECTION_JITTER_LOW_EXPANSION_STEPS,
            "direction_jitter_score_lower_percentile": DIRECTION_JITTER_SCORE_LOWER_PERCENTILE,
            "direction_jitter_score_upper_percentile": DIRECTION_JITTER_SCORE_UPPER_PERCENTILE,
            "broad_vibration_top_k": BROAD_VIBRATION_TOP_K,
            "broad_vibration_columns": BROAD_VIBRATION_COLUMNS,
        },
        # 旧推論ノートブック向けの最低限の互換キー。新推論では score_models を使用します。
        "model": primary_artifact["model"],
        "preprocess_pipeline": primary_artifact["preprocess_pipeline"],
        "feature_names": primary_artifact["feature_names"],
        "selected_source_columns": primary_artifact["selected_source_columns"],
        "expanded_feature_group_map": primary_artifact["expanded_feature_group_map"],
        "feature_weight_vector": primary_artifact["feature_weight_vector"],
    }, output_dir / "isolation_forest_feature_poc.joblib")
    print(f"saved outputs under: {output_dir}")

save_outputs(features_df, OUTPUT_DIR)

saved outputs under: /workspaces/forklift/outputs/anomaly_feature_poc


## 19. 次の検証観点

- 音が大きいところで `audio_rms` と `audio_high_freq_energy` が上がるか
- 瞬間的なx/y変化で `front_flow_change_amount_score` / `rear_flow_change_amount_score` が上がるか
- 近距離テクスチャ通過など局所的な変化で `flow_globality_score` が上がりすぎないか
- 旋回など長く続く変化で `flow_transience_score` が低く抑えられるか
- 接触・衝撃っぽい時刻で `flow_xy_impact_score` が上がるか
- `flow_high_change_cell_ratio` と `flow_event_duration_steps` を見て、広域性と持続時間の効き方を確認する
- 正常データだけで学習したとき、通常走行の一部が過剰に高スコアになっていないか
- front/rear の片側ピークと同時ピークを分けて解釈できるか

PoC で特徴量の妥当性が見えたら、次に速度・ステアリングなどのセンサー CSV を足して「走行状態ごとの正常分布」に近づけます。
